rag_pipeline_v4.svg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import json

BASE = '/content/drive/MyDrive/ddi_capstone'

# === DrugBank data ===
drug_info = pd.read_csv(f'{BASE}/full_drugbank_parsing/ogb_drug_info.csv')
drug_enzymes = pd.read_csv(f'{BASE}/full_drugbank_parsing/ogb_drug_enzymes.csv')
drug_targets = pd.read_csv(f'{BASE}/full_drugbank_parsing/ogb_drug_targets.csv')
drug_transporters = pd.read_csv(f'{BASE}/full_drugbank_parsing/ogb_drug_transporters.csv')
drug_carriers = pd.read_csv(f'{BASE}/full_drugbank_parsing/ogb_drug_carriers.csv')

# === Molecular data ===
drug_mapping = pd.read_csv(f'{BASE}/molecular/data/drug_mapping_smiles.csv')
drug_annotation = pd.read_csv(f'{BASE}/molecular/data/drug_annotation.csv')

# === Mappings ===
with open(f'{BASE}/full_drugbank_parsing/dbid_to_ogb_idx.json') as f:
    dbid_to_ogb = json.load(f)

print("=== drug_info ===")
print(drug_info.shape)
print(drug_info.columns.tolist())
print(drug_info.head(2))
print()

print("=== drug_enzymes ===")
print(drug_enzymes.shape)
print(drug_enzymes.columns.tolist())
print(drug_enzymes.head(2))
print()

print("=== drug_targets ===")
print(drug_targets.shape)
print(drug_targets.columns.tolist())
print()

print("=== drug_mapping ===")
print(drug_mapping.shape)
print(drug_mapping.columns.tolist())
print(drug_mapping.head(2))
print()

print("=== drug_annotation ===")
print(drug_annotation.shape)
print(drug_annotation.columns.tolist())
print(drug_annotation.head(2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== drug_info ===
(4266, 5)
['drugbank_id', 'name', 'type', 'smiles', 'ogb_idx']
  drugbank_id         name            type  \
0     DB00484  Brimonidine  small molecule   
1     DB01558   Bromazepam  small molecule   

                                   smiles  ogb_idx  
0        BrC1=C(NC2=NCCN2)C=CC2=NC=CN=C12      413  
1  BrC1=CC2=C(NC(=O)CN=C2C2=CC=CC=N2)C=C1     1318  

=== drug_enzymes ===
(5384, 4)
['drugbank_id', 'enzyme_name', 'uniprot_id', 'ogb_idx']
  drugbank_id          enzyme_name uniprot_id  ogb_idx
0     DB00006      Myeloperoxidase     P05164        4
1     DB00008  Cytochrome P450 1A2     P05177        6

=== drug_targets ===
(11556, 4)
['drugbank_id', 'target_name', 'uniprot_id', 'ogb_idx']

=== drug_mapping ===
(4267, 4)
['node idx', 'drug id', 'pubchem_cid', 'smiles']
   node idx  drug id  pubchem_cid smiles
0         0  DB00001        

In [ ]:
# ============================================
# STEP 2: Build Knowledge Base Chunks
# ============================================
#
# WHAT: Create a structured text document for each drug by joining
#        all DrugBank tables (info, enzymes, targets, transporters, carriers).
#
# WHY:  These chunks are the "documents" that will be embedded and stored
#       in our vector database (Step 3). When a user asks about a drug pair,
#       we'll retrieve the relevant chunks to give context to the LLM.
#
# INPUT:  DataFrames loaded in Step 1 (drug_info, drug_enzymes, etc.)
# OUTPUT: List of dicts, each with structured metadata + a 'text' field for embedding
# ============================================

def build_drug_chunks(drug_info, drug_enzymes, drug_targets, drug_transporters, drug_carriers):
    """Create one structured text document per drug combining all DrugBank sources.

    Each chunk contains both a 'text' field for embedding and structured metadata
    for filtering. The text is what ChromaDB searches against; the metadata is
    passed to the LLM as structured context.

    Args:
        drug_info: DataFrame with columns drugbank_id, ogb_idx, name, type, smiles.
        drug_enzymes: Long-format DataFrame with columns drugbank_id, enzyme_name.
        drug_targets: Long-format DataFrame with columns drugbank_id, target_name.
        drug_transporters: Long-format DataFrame with drugbank_id + transporter column.
        drug_carriers: Long-format DataFrame with drugbank_id + carrier column.

    Returns:
        List of dicts, one per drug, with keys: drug_id, ogb_idx, name, type,
        smiles, cyp_enzymes, all_enzymes, targets, transporters, carriers, text.
    """

    chunks = []

    for _, row in drug_info.iterrows():
        dbid = row['drugbank_id']
        ogb_idx = row['ogb_idx']

        # Retrieve associated enzymes for this drug
        enzymes = drug_enzymes[drug_enzymes['drugbank_id'] == dbid]['enzyme_name'].tolist()

        # Retrieve molecular targets
        targets = drug_targets[drug_targets['drugbank_id'] == dbid]['target_name'].tolist()

        # Retrieve transporters (column name may vary, so we use positional index)
        transporters = drug_transporters[drug_transporters['drugbank_id'] == dbid].iloc[:, 1].tolist() \
            if len(drug_transporters.columns) > 1 else []

        # Retrieve carriers
        carriers = drug_carriers[drug_carriers['drugbank_id'] == dbid].iloc[:, 1].tolist() \
            if len(drug_carriers.columns) > 1 else []

        # Filter CYP enzymes specifically (key for DDI mechanisms)
        cyp_enzymes = [e for e in enzymes if 'cytochrome' in e.lower() or 'cyp' in e.lower()]

        # Build the text that will be embedded
        # This is what ChromaDB will search against in Step 3
        text = f"""Drug: {row['name']} ({dbid})
Type: {row['type']}
CYP Enzymes: {', '.join(cyp_enzymes) if cyp_enzymes else 'None documented'}
Other Enzymes: {', '.join([e for e in enzymes if e not in cyp_enzymes]) if enzymes else 'None documented'}
Targets: {', '.join(targets[:10]) if targets else 'None documented'}
Transporters: {', '.join(transporters) if transporters else 'None documented'}
Carriers: {', '.join(carriers) if carriers else 'None documented'}"""

        # Structured chunk: 'text' is for embedding, everything else is metadata
        # Metadata will be useful for filtering and for passing to the LLM later
        chunk = {
            'drug_id': dbid,
            'ogb_idx': int(ogb_idx),
            'name': row['name'],
            'type': row['type'],
            'smiles': row['smiles'] if pd.notna(row['smiles']) else None,
            'cyp_enzymes': cyp_enzymes,
            'all_enzymes': enzymes,
            'targets': targets,
            'transporters': transporters,
            'carriers': carriers,
            'text': text
        }

        chunks.append(chunk)

    return chunks

# Build all chunks from the DataFrames loaded in Step 1
chunks = build_drug_chunks(drug_info, drug_enzymes, drug_targets, drug_transporters, drug_carriers)

# Quick sanity check
print(f"Total chunks created: {len(chunks)}")
print(f"Expected: {len(drug_info)} (one per drug in drug_info)")
print(f"\n--- Example chunk: {chunks[0]['name']} ---")
print(chunks[0]['text'])
print(f"\nCYP enzymes: {chunks[0]['cyp_enzymes']}")
print(f"Targets (first 5): {chunks[0]['targets'][:5]}")
print(f"Has SMILES: {chunks[0]['smiles'] is not None}")

Total chunks created: 4266
Expected: 4266 (one per drug in drug_info)

--- Example chunk: Brimonidine ---
Drug: Brimonidine (DB00484)
Type: small molecule
CYP Enzymes: None documented
Other Enzymes: Aldehyde oxidase
Targets: Alpha-2A adrenergic receptor, Alpha-2B adrenergic receptor, Alpha-2C adrenergic receptor
Transporters: None documented
Carriers: None documented

CYP enzymes: []
Targets (first 5): ['Alpha-2A adrenergic receptor', 'Alpha-2B adrenergic receptor', 'Alpha-2C adrenergic receptor']
Has SMILES: True


In [ ]:
# ============================================
# STEP 2b: Validate chunks with a CYP-rich drug
# ============================================
#
# WHY:  Brimonidine had no CYP enzymes. Let's check a drug we KNOW
#       has CYP interactions (e.g., Atorvastatin) to make sure
#       the CYP parsing works correctly before moving to Step 3.
# ============================================

# Find a known CYP-heavy drug
for chunk in chunks:
    if chunk['name'] == 'Atorvastatin':
        print(f"--- {chunk['name']} ---")
        print(chunk['text'])
        print(f"\nCYP enzymes: {chunk['cyp_enzymes']}")
        print(f"Targets (first 5): {chunk['targets'][:5]}")
        break

# Quick stats on CYP coverage
cyp_count = sum(1 for c in chunks if len(c['cyp_enzymes']) > 0)
print(f"\n--- Coverage Stats ---")
print(f"Drugs with CYP enzymes: {cyp_count}/{len(chunks)} ({cyp_count/len(chunks)*100:.1f}%)")
print(f"Drugs with targets: {sum(1 for c in chunks if len(c['targets']) > 0)}/{len(chunks)}")
print(f"Drugs with SMILES: {sum(1 for c in chunks if c['smiles'] is not None)}/{len(chunks)}")
print(f"Drugs with transporters: {sum(1 for c in chunks if len(c['transporters']) > 0)}/{len(chunks)}")

In [ ]:
# ============================================
# CHECKPOINT: Save chunks to Drive
# ============================================
#
# WHY:  If Colab disconnects, we don't want to re-run Steps 1-2.
#       Loading this JSON is instant and gets us back to where we left off.
#
# NEXT: In Step 3, we'll load this file and feed it into ChromaDB
#       to create the vector store for semantic search.
# ============================================

import json
import os

SAVE_PATH = f'{BASE}/rag_pipeline'
!mkdir -p "{SAVE_PATH}"

with open(f'{SAVE_PATH}/drug_chunks.json', 'w') as f:
    json.dump(chunks, f, indent=2)

print(f"Saved {len(chunks)} chunks to {SAVE_PATH}/drug_chunks.json")
print(f"File size: {os.path.getsize(f'{SAVE_PATH}/drug_chunks.json') / 1024:.1f} KB")

Saved 4266 chunks to /content/drive/MyDrive/ddi_capstone/rag_pipeline/drug_chunks.json
File size: 3492.5 KB


In [ ]:
# ============================================
# STEP 3: Create Vector Store with ChromaDB
# ============================================
#
# WHAT: Embed each drug chunk into a vector and store it in ChromaDB
#       so we can do semantic search (e.g., "CYP3A4 statin interaction").
#
# WHY:  When a user asks about a drug pair, we need to quickly find
#       the most relevant drug documents. Vector search lets us match
#       by meaning, not just keyword.
#
# INPUT:  drug_chunks.json from Step 2
# OUTPUT: ChromaDB collection with 4,266 embedded documents
#
# NOTE: This runs fine on CPU. Takes ~2-3 minutes for all drugs.
# ============================================

!pip install -q chromadb sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer
import json
import time

# Load chunks from checkpoint (safe if Colab reconnected)
SAVE_PATH = f'{BASE}/rag_pipeline'
with open(f'{SAVE_PATH}/drug_chunks.json', 'r') as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks from checkpoint")

# Initialize embedding model
# all-MiniLM-L6-v2: lightweight, fast, good general-purpose embeddings
# Produces 384-dimensional vectors
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

# Initialize ChromaDB (persistent storage in Drive so it survives restarts)
CHROMA_PATH = f'{SAVE_PATH}/chromadb'
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Create (or get) the collection
# If it already exists from a previous run, this will reuse it
collection = client.get_or_create_collection(
    name="ddi_drugs",
    metadata={"description": "DrugBank drug profiles for DDI RAG system"}
)

# Check if already populated (skip re-indexing on Colab restart)
if collection.count() >= len(chunks):
    print(f"Collection already has {collection.count()} documents. Skipping indexing.")
else:
    print(f"Indexing {len(chunks)} drug chunks...")
    start = time.time()

    # Process in batches of 100 (ChromaDB handles batches efficiently)
    BATCH_SIZE = 100
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]

        # Extract texts for this batch
        texts = [c['text'] for c in batch]

        # Generate embeddings
        embeddings = embedder.encode(texts).tolist()

        # Prepare metadata (ChromaDB only accepts str, int, float, bool)
        metadatas = [{
            'drug_id': c['drug_id'],
            'ogb_idx': c['ogb_idx'],
            'name': c['name'],
            'type': c['type'],
            'has_cyp': len(c['cyp_enzymes']) > 0,
            'cyp_list': ', '.join(c['cyp_enzymes']) if c['cyp_enzymes'] else 'none',
            'n_targets': len(c['targets']),
            'n_enzymes': len(c['all_enzymes']),
            'has_smiles': c['smiles'] is not None
        } for c in batch]

        # IDs must be unique strings
        ids = [c['drug_id'] for c in batch]

        # Add to collection
        collection.add(
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )

        # Progress
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  Indexed {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks...")

    elapsed = time.time() - start
    print(f"\nDone. Indexed {collection.count()} documents in {elapsed:.1f}s")
    print(f"ChromaDB persisted to: {CHROMA_PATH}")

Loaded 4266 chunks from checkpoint
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded
Collection already has 4266 documents. Skipping indexing.


In [ ]:
# ============================================
# STEP 3b: Test retrieval with sample queries
# ============================================
#
# WHY:  Before building the full RAG pipeline, we need to verify
#       that semantic search returns relevant results.
#       We test with queries similar to what the LLM will generate
#       when a user asks about a drug interaction.
#
# NEXT: In Step 4, we'll combine retrieval results with EXAI model
#       outputs and send everything to Claude API.
# ============================================

def search_drugs(query, n_results=5):
    """Search the vector store and display results."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    print(f"Query: '{query}'")
    print(f"{'='*60}")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"\n#{i+1} [{meta['name']}] (distance: {dist:.4f})")
        print(f"   CYP: {meta['cyp_list']}")
        print(f"   Targets: {meta['n_targets']} | Enzymes: {meta['n_enzymes']}")
    print()

# Test 1: Search by enzyme pathway
search_drugs("CYP3A4 substrate statin metabolism")

# Test 2: Search by drug name
search_drugs("clarithromycin antibiotic CYP inhibitor")

# Test 3: Search relevant to one of your 5 validation pairs
search_drugs("benzodiazepine CYP3A4 alprazolam")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 35.8MiB/s]


Query: 'CYP3A4 substrate statin metabolism'

#1 [Cymarin] (distance: 0.8939)
   CYP: none
   Targets: 0 | Enzymes: 0

#2 [Synthetic Conjugated Estrogens, B] (distance: 0.9556)
   CYP: Cytochrome P450 3A4
   Targets: 1 | Enzymes: 1

#3 [Sisomicin] (distance: 0.9607)
   CYP: none
   Targets: 0 | Enzymes: 0

#4 [Protirelin] (distance: 0.9611)
   CYP: none
   Targets: 1 | Enzymes: 0

#5 [Cyproterone acetate] (distance: 0.9750)
   CYP: Cytochrome P450 3A4
   Targets: 2 | Enzymes: 2

Query: 'clarithromycin antibiotic CYP inhibitor'

#1 [Clarithromycin] (distance: 0.6357)
   CYP: Cytochrome P450 3A4, Cytochrome P450 3A5
   Targets: 4 | Enzymes: 2

#2 [Clometocillin] (distance: 0.7465)
   CYP: none
   Targets: 0 | Enzymes: 0

#3 [Cyclacillin] (distance: 0.7519)
   CYP: none
   Targets: 4 | Enzymes: 0

#4 [Biapenem] (distance: 0.7528)
   CYP: none
   Targets: 1 | Enzymes: 0

#5 [Pirlimycin] (distance: 0.7833)
   CYP: none
   Targets: 0 | Enzymes: 0

Query: 'benzodiazepine CYP3A4 alprazolam'

#1

In [ ]:
# ============================================
# STEP 3c: Hybrid Retrieval (Direct Lookup + Semantic Search)
# ============================================
#
# WHY:  Pure semantic search misses known drugs because the general-purpose
#       embedding model (all-MiniLM-L6-v2) doesn't understand pharmacology well.
#       For DDI queries we almost always know the drug names, so we use direct
#       metadata lookup first (exact match), then semantic search for additional
#       context (e.g., drugs with similar CYP profiles).
#
# This is the retrieval strategy used in the final RAG pipeline (Step 5).
# ============================================

def retrieve_drug_context(drug_name_a, drug_name_b, collection, n_similar=3):
    """Hybrid retrieval: direct metadata lookup + semantic search for related drugs.

    Pure semantic search misses known drugs because general-purpose embedding
    models don't understand pharmacology well. Direct lookup by name is always
    exact; semantic search adds related drugs for broader mechanistic context.

    Args:
        drug_name_a (str): Name of the first drug (must match 'name' metadata
            field exactly as stored at indexing time).
        drug_name_b (str): Name of the second drug.
        collection: ChromaDB collection with embedded drug chunks.
        n_similar (int): Number of semantically similar drugs to retrieve
            beyond the two query drugs. Defaults to 3.

    Returns:
        dict with keys:
            'drug_a': {'document': str, 'metadata': dict} or None if not found.
            'drug_b': {'document': str, 'metadata': dict} or None if not found.
            'similar_drugs': list of {'document': str, 'metadata': dict}.
    """

    # --- PART 1: Direct lookup by drug name ---
    # Exact match on metadata field — always succeeds if the drug is indexed
    direct_a = collection.get(
        where={"name": drug_name_a},
        include=["documents", "metadatas"]
    )
    direct_b = collection.get(
        where={"name": drug_name_b},
        include=["documents", "metadatas"]
    )

    # Warn on name mismatch — most common failure mode is capitalization
    if not direct_a['documents']:
        print(f"Warning: '{drug_name_a}' not found in collection. "
              "Check spelling and capitalization against the index.")
    if not direct_b['documents']:
        print(f"Warning: '{drug_name_b}' not found in collection. "
              "Check spelling and capitalization against the index.")

    # --- PART 2: Semantic search for related drugs ---
    # Finds drugs with similar mechanisms/enzymes that may inform the explanation.
    # We over-fetch by 2 to ensure n_similar results after filtering out the
    # query drugs themselves, which often appear at the top of semantic results.
    query = f"{drug_name_a} {drug_name_b} interaction mechanism"
    similar = collection.query(
        query_texts=[query],
        n_results=n_similar + 2
    )

    semantic_results = []
    for doc, meta in zip(similar['documents'][0], similar['metadatas'][0]):
        if meta['name'] not in [drug_name_a, drug_name_b]:
            semantic_results.append({'document': doc, 'metadata': meta})
        if len(semantic_results) >= n_similar:
            break

    return {
        'drug_a': {
            'document': direct_a['documents'][0] if direct_a['documents'] else None,
            'metadata': direct_a['metadatas'][0] if direct_a['metadatas'] else None
        },
        'drug_b': {
            'document': direct_b['documents'][0] if direct_b['documents'] else None,
            'metadata': direct_b['metadatas'][0] if direct_b['metadatas'] else None
        },
        'similar_drugs': semantic_results
    }


# --- Test with one of the 5 validation pairs ---
context = retrieve_drug_context("Atorvastatin", "Clarithromycin", collection)

print("=== DRUG A ===")
print(context['drug_a']['document'])
print("\n=== DRUG B ===")
print(context['drug_b']['document'])
print("\n=== SIMILAR DRUGS (for additional context) ===")
for i, s in enumerate(context['similar_drugs']):
    print(f"\n#{i+1} {s['metadata']['name']}")
    print(f"   CYP: {s['metadata'].get('cyp_list', 'N/A')}")

=== DRUG A ===
Drug: Atorvastatin (DB01076)
Type: small molecule
CYP Enzymes: Cytochrome P450 3A4, Cytochrome P450 3A5, Cytochrome P450 3A7, Cytochrome P450 2C8, Cytochrome P450 2D6, Cytochrome P450 2C9, Cytochrome P450 2C19, Cytochrome P450 2B6
Other Enzymes: UDP-glucuronosyltransferase 1A1, UDP-glucuronosyltransferase 1A3
Targets: 3-hydroxy-3-methylglutaryl-coenzyme A reductase, Dipeptidyl peptidase 4, Aryl hydrocarbon receptor, Histone deacetylase 2, Nuclear receptor subfamily 1 group I member 3
Transporters: ATP-dependent translocase ABCB1, Solute carrier organic anion transporter family member 1A2, Solute carrier organic anion transporter family member 1B1, ATP-binding cassette sub-family C member 4, ATP-binding cassette sub-family C member 5, Multidrug resistance-associated protein 1, Solute carrier organic anion transporter family member 2B1, Solute carrier organic anion transporter family member 1B3, ATP-binding cassette sub-family C member 2, Bile salt export pump
Carriers: Al

In [ ]:
# ============================================
# CHECKPOINT: Save retrieval function and test results
# ============================================
#
# WHY:  The ChromaDB is already persisted in Drive from Step 3.
#       Here we save a quick test result so we can verify the
#       retrieval still works after a Colab restart.
#
# TO RESTORE after restart, you only need:
#   1. Load chunks: json.load('rag_pipeline/drug_chunks.json')
#   2. Load ChromaDB: PersistentClient(path='rag_pipeline/chromadb')
#   3. Load embedder: SentenceTransformer('all-MiniLM-L6-v2')
#   4. Get collection: client.get_collection('ddi_drugs')
# ============================================

# Save a restore snippet as a .py file for convenience
restore_code = """
# === QUICK RESTORE after Colab restart ===
import json
import chromadb
from sentence_transformers import SentenceTransformer

BASE = '/content/drive/MyDrive/ddi_capstone'
SAVE_PATH = f'{BASE}/rag_pipeline'

# Load chunks
with open(f'{SAVE_PATH}/drug_chunks.json', 'r') as f:
    chunks = json.load(f)

# Load ChromaDB
client = chromadb.PersistentClient(path=f'{SAVE_PATH}/chromadb')
collection = client.get_collection('ddi_drugs')

# Load embedder
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Restored: {len(chunks)} chunks, {collection.count()} indexed documents")
"""

with open(f'{SAVE_PATH}/restore_session.py', 'w') as f:
    f.write(restore_code)

print(f"Restore script saved to {SAVE_PATH}/restore_session.py")
print("ChromaDB already persisted in Drive from Step 3")

Restore script saved to /content/drive/MyDrive/ddi_capstone/rag_pipeline/restore_session.py
ChromaDB already persisted in Drive from Step 3


In [ ]:
# ============================================
# STEP 4: Load EXAI model results
# ============================================
#
# WHAT: Load the explainability results from your 7 trained models
#       (MLP, GraphSAGE x2, GAT x4) so we can pass them to Claude
#       alongside the DrugBank context from ChromaDB.
#
# WHY:  The RAG system's unique value is combining pharmacological
#       knowledge (from retrieval) with model explanations (from EXAI).
#       This is what differentiates it from a generic drug interaction lookup.
#
# INPUT:  JSON result files saved during your model training/analysis
# OUTPUT: A function that returns EXAI explanations for any drug pair
# ============================================

import json

# Load all EXAI results
with open(f'{BASE}/MLP_baseline/models/ig_results.json', 'r') as f:
    mlp_ig = json.load(f)

with open(f'{BASE}/graphsage/perturbation_results.json', 'r') as f:
    graphsage_perturbation = json.load(f)

with open(f'{BASE}/graphsage/ig_results.json', 'r') as f:
    graphsage_ig = json.load(f)

with open(f'{BASE}/GAT_ablation_h128/results/exai_results.json', 'r') as f:
    gat_exai = json.load(f)

with open(f'{BASE}/GAT_ablation_h128/results/ig_results.json', 'r') as f:
    gat_ig = json.load(f)

with open(f'{BASE}/GAT_ablation_h128/results/perturbation_results.json', 'r') as f:
    gat_perturbation = json.load(f)

# Quick look at the structure of each
print("=== MLP IG results ===")
print(f"Type: {type(mlp_ig)}")
if isinstance(mlp_ig, dict):
    print(f"Keys: {list(mlp_ig.keys())[:5]}")
elif isinstance(mlp_ig, list):
    print(f"Length: {len(mlp_ig)}")
    print(f"First item keys: {list(mlp_ig[0].keys()) if mlp_ig else 'empty'}")

print(f"\n=== GraphSAGE perturbation ===")
print(f"Type: {type(graphsage_perturbation)}")
if isinstance(graphsage_perturbation, dict):
    print(f"Keys: {list(graphsage_perturbation.keys())[:5]}")
elif isinstance(graphsage_perturbation, list):
    print(f"Length: {len(graphsage_perturbation)}")
    print(f"First item keys: {list(graphsage_perturbation[0].keys()) if graphsage_perturbation else 'empty'}")

print(f"\n=== GAT EXAI results ===")
print(f"Type: {type(gat_exai)}")
if isinstance(gat_exai, dict):
    print(f"Keys: {list(gat_exai.keys())[:5]}")
elif isinstance(gat_exai, list):
    print(f"Length: {len(gat_exai)}")
    print(f"First item keys: {list(gat_exai[0].keys()) if gat_exai else 'empty'}")

print(f"\n=== GAT perturbation ===")
print(f"Type: {type(gat_perturbation)}")
if isinstance(gat_perturbation, dict):
    print(f"Keys: {list(gat_perturbation.keys())[:5]}")

=== MLP IG results ===
Type: <class 'dict'>
Keys: ['Terfenadine + Alprazolam', 'Nilotinib + Dacomitinib', 'Palonosetron + Clomipramine', 'Flunitrazepam + Alprazolam', 'Clomipramine + Atomoxetine']

=== GraphSAGE perturbation ===
Type: <class 'dict'>
Keys: ['topo', 'feat']

=== GAT EXAI results ===
Type: <class 'dict'>
Keys: ['attention', 'concordance']

=== GAT perturbation ===
Type: <class 'dict'>
Keys: ['Terfenadine + Alprazolam', 'Nilotinib + Dacomitinib', 'Palonosetron + Clomipramine', 'Flunitrazepam + Alprazolam', 'Clomipramine + Atomoxetine']


In [ ]:
# ============================================
# STEP 4b: Explore EXAI result structure (deeper)
# ============================================
#
# WHY:  We need to understand the exact structure of each EXAI result
#       to build the function that extracts explanations per drug pair.
#       Each model stores results differently.
# ============================================

# Use one consistent pair to explore all models
PAIR = 'Terfenadine + Alprazolam'

# --- MLP IG ---
print("=== MLP IG ===")
mlp_pair = mlp_ig[PAIR]
print(f"Keys: {list(mlp_pair.keys())}")
for k, v in mlp_pair.items():
    if isinstance(v, list):
        print(f"  {k}: list[{len(v)}], first: {v[:3]}")
    elif isinstance(v, dict):
        print(f"  {k}: dict with keys {list(v.keys())[:5]}")
    else:
        print(f"  {k}: {v}")

# --- GraphSAGE perturbation ---
print("\n=== GraphSAGE perturbation (topo) ===")
gs_topo = graphsage_perturbation['topo']
print(f"Keys: {list(gs_topo.keys())[:5]}")
if PAIR in gs_topo:
    gs_pair = gs_topo[PAIR]
    print(f"Pair keys: {list(gs_pair.keys())}")
    for k, v in gs_pair.items():
        if isinstance(v, list):
            print(f"  {k}: list[{len(v)}], first item: {v[0] if v else 'empty'}")
        else:
            print(f"  {k}: {type(v).__name__} = {str(v)[:100]}")

print("\n=== GraphSAGE perturbation (feat) ===")
gs_feat = graphsage_perturbation['feat']
print(f"Keys: {list(gs_feat.keys())[:5]}")

# --- GAT EXAI (attention) ---
print("\n=== GAT EXAI attention ===")
gat_att = gat_exai['attention']
print(f"Keys: {list(gat_att.keys())[:5]}")
first_model = list(gat_att.keys())[0]
print(f"First model '{first_model}' keys: {list(gat_att[first_model].keys())[:5]}")

# --- GAT perturbation ---
print("\n=== GAT perturbation ===")
gat_pair = gat_perturbation[PAIR]
print(f"Keys: {list(gat_pair.keys())}")
for k, v in gat_pair.items():
    if isinstance(v, dict):
        print(f"  {k}: dict with keys {list(v.keys())[:5]}")
    elif isinstance(v, list):
        print(f"  {k}: list[{len(v)}]")
    else:
        print(f"  {k}: {v}")

# --- GAT IG ---
print("\n=== GAT IG ===")
print(f"Keys: {list(gat_ig.keys())[:5]}")
first_key = list(gat_ig.keys())[0]
print(f"First key '{first_key}': {type(gat_ig[first_key])}")
if isinstance(gat_ig[first_key], dict):
    print(f"  Sub-keys: {list(gat_ig[first_key].keys())[:5]}")

=== MLP IG ===
Keys: ['prediction', 'drug_a', 'drug_b', 'id_a', 'id_b', 'shared_enzymes', 'shared_targets', 'drug_a_top_bits', 'drug_b_top_bits', 'drug_a_top_values', 'drug_b_top_values']
  prediction: 0.9078190326690674
  drug_a: Terfenadine
  drug_b: Alprazolam
  id_a: 279
  id_b: 338
  shared_enzymes: list[3], first: ['Cytochrome P450 3A4', 'Cytochrome P450 3A5', 'Cytochrome P450 3A7']
  shared_targets: list[0], first: []
  drug_a_top_bits: list[10], first: [1437, 808, 925]
  drug_b_top_bits: list[10], first: [168, 1571, 825]
  drug_a_top_values: list[10], first: [-0.7036470770835876, 0.7024990916252136, -0.46255019307136536]
  drug_b_top_values: list[10], first: [0.3372424244880676, 0.3308171033859253, 0.3296908736228943]

=== GraphSAGE perturbation (topo) ===
Keys: ['Terfenadine + Alprazolam', 'Nilotinib + Dacomitinib', 'Palonosetron + Clomipramine', 'Flunitrazepam + Alprazolam', 'Clomipramine + Atomoxetine']
Pair keys: ['orig_score', 'neighbors']
  orig_score: float = 1.864409565

In [ ]:
# ============================================
# STEP 4c: Build Unified ExAI Context Extractor
# ============================================
#
# WHAT: A single function that, given a drug pair name, collects
#       all available ExAI explanations across all 7 models.
#
# WHY:  When we send context to Claude (Step 5), we need a clean
#       summary of what each model says about the interaction.
#       This function bridges the gap between raw JSON files
#       and the structured prompt we build in Step 5.
#
# NOTE: Currently limited to the 5 validation pairs. For new pairs,
#       model inference would need to be re-run (documented future work).
# ============================================

VALID_PAIRS = [
    'Terfenadine + Alprazolam',
    'Nilotinib + Dacomitinib',
    'Palonosetron + Clomipramine',
    'Flunitrazepam + Alprazolam',
    'Clomipramine + Atomoxetine'
]


def get_exai_context(pair_name, mlp_ig, graphsage_perturbation, graphsage_ig,
                     gat_exai, gat_ig, gat_perturbation):
    """Collect and format all ExAI explanations for a drug pair across all 7 models.

    Aggregates MLP Integrated Gradients, GraphSAGE perturbation and IG,
    and GAT attention weights, perturbation, and IG into a single formatted
    string ready to insert into the Claude API prompt.

    Args:
        pair_name (str): Drug pair string, e.g. 'Terfenadine + Alprazolam'.
            Must be one of VALID_PAIRS (the 5 pre-analyzed validation pairs).
        mlp_ig (dict): MLP IG results keyed by pair name.
        graphsage_perturbation (dict): GraphSAGE perturbation results,
            keyed by variant ('topo', 'feat') then pair name.
        graphsage_ig (dict): GraphSAGE IG results keyed by pair name.
        gat_exai (dict): GAT attention + concordance results.
        gat_ig (dict): GAT IG results keyed by pair name.
        gat_perturbation (dict): GAT perturbation results keyed by pair name.

    Returns:
        str: Formatted multi-section string with all model explanations,
        or an error message if pair_name is not in VALID_PAIRS.
    """
    if pair_name not in VALID_PAIRS:
        return (
            f"No ExAI results available for '{pair_name}'. "
            f"Pre-analyzed pairs: {VALID_PAIRS}"
        )

    sections = []

    # --- 1. MLP: Integrated Gradients on molecular fingerprints ---
    # MLP captures pharmacokinetic DDIs via molecular structure alone.
    # Shared CYP enzymes indicate metabolic pathway overlap.
    mlp = mlp_ig.get(pair_name, {})
    if mlp:
        sections.append(
            f"## MLP Baseline (Molecular Fingerprints)\n"
            f"- Prediction score: {mlp.get('prediction', 'N/A'):.4f}\n"
            f"- Shared CYP enzymes: "
            f"{', '.join(mlp['shared_enzymes']) if mlp.get('shared_enzymes') else 'None'}\n"
            f"- Shared targets: "
            f"{', '.join(mlp['shared_targets']) if mlp.get('shared_targets') else 'None'}\n"
            f"- Drug A top fingerprint bits: {mlp.get('drug_a_top_bits', [])[:5]}\n"
            f"- Drug B top fingerprint bits: {mlp.get('drug_b_top_bits', [])[:5]}"
        )

    # --- 2. GraphSAGE: Perturbation analysis (topology-only and feature-enhanced) ---
    # Impact = score change when neighbor is masked from the graph.
    # CYP overlap explains why a neighbor is influential.
    for variant in ['topo', 'feat']:
        gs_variant = graphsage_perturbation.get(variant, {})
        gs_data = gs_variant.get(pair_name)
        if not gs_data:
            continue

        neighbor_summary = []
        for n in gs_data.get('neighbors', [])[:5]:
            if not isinstance(n, dict):
                continue
            cyp_str = ', '.join(n['cyp_enzymes']) if n.get('cyp_enzymes') else 'none'
            neighbor_summary.append(
                f"  - {n.get('name', '?')} "
                f"(impact: {n.get('impact', 0):.4f}, CYP: {cyp_str})"
            )

        sections.append(
            f"## GraphSAGE ({variant}) — Perturbation Analysis\n"
            f"- Original score: {gs_data.get('orig_score', 'N/A'):.4f}\n"
            f"- Top 5 influential neighbors:\n"
            + ("\n".join(neighbor_summary) if neighbor_summary else "  None available")
        )

    # --- 3. GraphSAGE IG (feature-enhanced model only) ---
    # IG attributes the prediction to specific Morgan fingerprint bits.
    gs_ig_pair = graphsage_ig.get(pair_name)
    if gs_ig_pair and isinstance(gs_ig_pair, dict):
        sections.append(
            f"## GraphSAGE (feat) — Integrated Gradients\n"
            f"- Available keys: {list(gs_ig_pair.keys())[:5]}"
        )

    # --- 4. GAT: Attention weights (all 4 variants) ---
    # Attention weights show which neighbors the model focused on during encoding.
    # Note: low concordance with perturbation (~8%) is expected — complementary methods.
    gat_att_pair = gat_exai.get('attention', {}).get(pair_name, {})
    for model_name, att_data in gat_att_pair.items():
        if isinstance(att_data, dict):
            sections.append(
                f"## {model_name} — Attention Weights\n"
                f"- Available metrics: {list(att_data.keys())[:3]}"
            )

    # --- 5. GAT: Perturbation analysis (all 4 variants) ---
    gat_pert_pair = gat_perturbation.get(pair_name, {})
    for model_name, neighbors in gat_pert_pair.items():
        if not isinstance(neighbors, list):
            continue

        neighbor_summary = []
        for n in neighbors[:5]:
            if not isinstance(n, dict):
                continue
            name   = n.get('name', n.get('neighbor_idx', '?'))
            impact = n.get('impact', n.get('score_change', 0))
            cyp_str = ', '.join(n['cyp_enzymes']) if n.get('cyp_enzymes') else 'none'
            neighbor_summary.append(
                f"  - {name} (impact: {impact:.4f}, CYP: {cyp_str})"
            )

        sections.append(
            f"## {model_name} — Perturbation Analysis\n"
            f"- Top 5 influential neighbors:\n"
            + ("\n".join(neighbor_summary) if neighbor_summary else "  None available")
        )

    # --- 6. GAT: Integrated Gradients (feature-enhanced variants only) ---
    # Skip connections in GAT Skip+Feat produce ~300× stronger attributions
    # than GAT Base+Feat due to enhanced gradient flow to fingerprint input.
    gat_ig_pair = gat_ig.get(pair_name, {})
    for model_name, ig_data in gat_ig_pair.items():
        if isinstance(ig_data, dict):
            sections.append(
                f"## {model_name} — Integrated Gradients\n"
                f"- Available keys: {list(ig_data.keys())[:5]}"
            )

    return '\n\n'.join(sections)


# --- Test: full ExAI context for one validation pair ---
pair = 'Terfenadine + Alprazolam'
exai_text = get_exai_context(
    pair, mlp_ig, graphsage_perturbation, graphsage_ig,
    gat_exai, gat_ig, gat_perturbation
)
print(f"ExAI Context for: {pair}")
print("=" * 60)
print(exai_text)
print(f"\nTotal length: {len(exai_text)} characters")

EXAI Context for: Terfenadine + Alprazolam
## MLP Baseline (Molecular Fingerprints)
- Prediction score: 0.9078
- Shared CYP enzymes: Cytochrome P450 3A4, Cytochrome P450 3A5, Cytochrome P450 3A7
- Shared targets: None
- Drug A top important fingerprint bits: [1437, 808, 925, 1261, 1610]
- Drug B top important fingerprint bits: [168, 1571, 825, 982, 843]

## GraphSAGE (topo) - Perturbation Analysis
- Original score: 1.8644
- Top 5 influential neighbors:
  - Adipiplon (impact: 0.0107, CYP: none)
  - Oxiracetam (impact: 0.0099, CYP: none)
  - Ethchlorvynol (impact: 0.0097, CYP: none)
  - Nomifensine (impact: 0.0097, CYP: none)
  - Meprobamate (impact: 0.0096, CYP: none)

## GraphSAGE (feat) - Perturbation Analysis
- Original score: 1.8336
- Top 5 influential neighbors:
  - Diacerein (impact: 0.0017, CYP: none)
  - Adipiplon (impact: 0.0017, CYP: none)
  - Rhein (impact: 0.0016, CYP: none)
  - Cabozantinib (impact: 0.0016, CYP: CYP2C9, CYP3A4)
  - Pyrithyldione (impact: 0.0014, CYP: none)


In [ ]:
# ============================================
# STEP 4d: Fix GAT perturbation parsing + normalize scores
# ============================================
#
# WHY:  Two issues found in Step 4c output:
#       1. GAT perturbation neighbors show indices instead of names
#       2. GAT perturbation impacts show 0.0000 (wrong key)
#       We also normalize GraphSAGE scores to 0-1 using sigmoid.
#
# INPUT:  Raw GAT perturbation JSON + drug_info for name lookup
# OUTPUT: Updated get_exai_context function with fixes
# ============================================

import math

# First, let's inspect the actual structure of a GAT perturbation entry
pair = 'Terfenadine + Alprazolam'
print("=== GAT perturbation raw structure ===")
print(f"Top-level keys: {list(gat_perturbation[pair].keys())}")

# Look at one model's first neighbor entry
for model_name in ['GAT_base', 'GAT_skip', 'GAT_base_feat', 'GAT_skip_feat']:
    neighbors = gat_perturbation[pair][model_name]
    if neighbors:
        print(f"\n{model_name} - first neighbor:")
        print(f"  Type: {type(neighbors[0])}")
        if isinstance(neighbors[0], dict):
            print(f"  Keys: {list(neighbors[0].keys())}")
            print(f"  Values: {neighbors[0]}")
        elif isinstance(neighbors[0], list):
            print(f"  Length: {len(neighbors[0])}")
            print(f"  Content: {neighbors[0]}")
        else:
            print(f"  Value: {neighbors[0]}")

=== GAT perturbation raw structure ===
Top-level keys: ['GAT_base', 'GAT_skip', 'GAT_base_feat', 'GAT_skip_feat']

GAT_base - first neighbor:
  Type: <class 'dict'>
  Keys: ['neighbor_idx', 'neighbor_name', 'baseline_score', 'perturbed_score', 'delta', 'abs_delta', 'is_drug_b']
  Values: {'neighbor_idx': 304, 'neighbor_name': 'Norepinephrine', 'baseline_score': 1.7062454223632812, 'perturbed_score': 1.7065544128417969, 'delta': -0.000308990478515625, 'abs_delta': 0.000308990478515625, 'is_drug_b': False}

GAT_skip - first neighbor:
  Type: <class 'dict'>
  Keys: ['neighbor_idx', 'neighbor_name', 'baseline_score', 'perturbed_score', 'delta', 'abs_delta', 'is_drug_b']
  Values: {'neighbor_idx': 1934, 'neighbor_name': 'Coltuximab ravtansine', 'baseline_score': 1.713699460029602, 'perturbed_score': 1.7129663228988647, 'delta': 0.0007331371307373047, 'abs_delta': 0.0007331371307373047, 'is_drug_b': False}

GAT_base_feat - first neighbor:
  Type: <class 'dict'>
  Keys: ['neighbor_idx', 'neig

In [ ]:
# ============================================
# STEP 4e: Fixed EXAI context extractor
# ============================================
#
# FIXES from Step 4d inspection:
#   1. GAT perturbation: use 'neighbor_name' and 'abs_delta' (not 'name'/'impact')
#   2. All scores normalized to 0-1 via sigmoid where needed
#   3. GAT IG: extract actual top bits and values
#   4. GAT attention: extract actual top neighbor names and alphas
#
# INPUT:  Same EXAI JSON dicts from Step 4
# OUTPUT: Clean formatted string for LLM prompt (Step 5)
# ============================================

import math

def sigmoid(x):
    """Convert logit to probability."""
    return 1 / (1 + math.exp(-x))


def get_exai_context(pair_name, mlp_ig, graphsage_perturbation, graphsage_ig,
                     gat_exai, gat_ig, gat_perturbation):
    """
    Collect all EXAI explanations for a drug pair across all 7 models.
    Returns a formatted string ready to insert into the LLM prompt.
    """

    if pair_name not in VALID_PAIRS:
        return f"No EXAI results available for '{pair_name}'. Only pre-analyzed pairs: {VALID_PAIRS}"

    sections = []

    # ---- 1. MLP: Integrated Gradients on molecular fingerprints ----
    mlp = mlp_ig[pair_name]
    sections.append(f"""## MLP Baseline (Molecular Fingerprints)
- Prediction score: {mlp['prediction']:.4f}
- Shared CYP enzymes: {', '.join(mlp['shared_enzymes']) if mlp['shared_enzymes'] else 'None'}
- Shared targets: {', '.join(mlp['shared_targets']) if mlp['shared_targets'] else 'None'}
- Drug A ({mlp['drug_a']}) top fingerprint bits: {mlp['drug_a_top_bits'][:5]}
- Drug B ({mlp['drug_b']}) top fingerprint bits: {mlp['drug_b_top_bits'][:5]}""")

    # ---- 2. GraphSAGE: Perturbation (topo + feat) ----
    for variant in ['topo', 'feat']:
        gs_data = graphsage_perturbation[variant][pair_name]
        score_prob = sigmoid(gs_data['orig_score'])
        top_neighbors = gs_data['neighbors'][:5]
        neighbor_lines = []
        for n in top_neighbors:
            cyp_str = ', '.join(n['cyp_enzymes']) if n.get('cyp_enzymes') else 'none'
            neighbor_lines.append(f"  - {n['name']} (impact: {n['impact']:.4f}, CYP: {cyp_str})")

        sections.append(f"""## GraphSAGE ({variant}) - Perturbation Analysis
- Prediction score: {score_prob:.4f} (logit: {gs_data['orig_score']:.4f})
- Top 5 influential neighbors:
{chr(10).join(neighbor_lines)}""")

    # ---- 3. GraphSAGE IG (feat model) ----
    gs_ig_pair = graphsage_ig.get(pair_name, {})
    if gs_ig_pair and isinstance(gs_ig_pair, dict):
        for drug_key in ['drug_a', 'drug_b']:
            if drug_key in gs_ig_pair:
                ig_data = gs_ig_pair[drug_key]
                if isinstance(ig_data, dict) and 'top_bits' in ig_data:
                    sections.append(f"""## GraphSAGE (feat) - Integrated Gradients ({drug_key})
- Top fingerprint bits: {ig_data.get('top_bits', [])[:5]}
- Top values: {[f'{v:.4f}' for v in ig_data.get('top_values', [])[:5]]}""")

    # ---- 4. GAT: Attention weights (4 variants) ----
    gat_att_pair = gat_exai['attention'].get(pair_name, {})
    for model_name, att_data in gat_att_pair.items():
        if isinstance(att_data, dict):
            # Extract top attended neighbors with names and alpha values
            top_a = att_data.get('drug_a_top10', [])[:5]
            alphas_a = att_data.get('drug_a_top10_alphas', [])[:5]
            top_b = att_data.get('drug_b_top10', [])[:5]
            alphas_b = att_data.get('drug_b_top10_alphas', [])[:5]

            lines = []
            if top_a:
                for name, alpha in zip(top_a, alphas_a):
                    a_val = f"{alpha:.4f}" if isinstance(alpha, (int, float)) else str(alpha)
                    lines.append(f"  - Drug A → {name} (alpha: {a_val})")
            if top_b:
                for name, alpha in zip(top_b, alphas_b):
                    b_val = f"{alpha:.4f}" if isinstance(alpha, (int, float)) else str(alpha)
                    lines.append(f"  - Drug B → {name} (alpha: {b_val})")

            sections.append(f"""## {model_name} - Attention Weights
- Top attended neighbors:
{chr(10).join(lines) if lines else '  No attention data available'}""")

    # ---- 5. GAT: Perturbation (4 variants) ----
    gat_pert_pair = gat_perturbation[pair_name]
    for model_name, neighbors in gat_pert_pair.items():
        if isinstance(neighbors, list) and neighbors:
            score_prob = sigmoid(neighbors[0].get('baseline_score', 0))
            top_5 = neighbors[:5]
            neighbor_lines = []
            for n in top_5:
                name = n.get('neighbor_name', n.get('neighbor_idx', '?'))
                delta = n.get('abs_delta', 0)
                neighbor_lines.append(f"  - {name} (abs_delta: {delta:.6f})")

            sections.append(f"""## {model_name} - Perturbation Analysis
- Prediction score: {score_prob:.4f}
- Top 5 influential neighbors:
{chr(10).join(neighbor_lines)}""")

    # ---- 6. GAT: Integrated Gradients (feat models only) ----
    gat_ig_pair = gat_ig.get(pair_name, {})
    for model_name, ig_data in gat_ig_pair.items():
        if isinstance(ig_data, dict):
            top_a_bits = ig_data.get('drug_a_top_bits', [])[:5]
            top_a_vals = ig_data.get('drug_a_top_values', [])[:5]
            top_b_bits = ig_data.get('drug_b_top_bits', [])[:5]
            top_b_vals = ig_data.get('drug_b_top_values', [])[:5]

            sections.append(f"""## {model_name} - Integrated Gradients
- Drug A top bits: {top_a_bits}, values: {[f'{v:.4f}' for v in top_a_vals]}
- Drug B top bits: {top_b_bits}, values: {[f'{v:.4f}' for v in top_b_vals]}""")

    return '\n\n'.join(sections)


# --- TEST with fixed function ---
pair = 'Terfenadine + Alprazolam'
exai_text = get_exai_context(pair, mlp_ig, graphsage_perturbation, graphsage_ig,
                              gat_exai, gat_ig, gat_perturbation)
print(f"EXAI Context for: {pair}")
print(f"{'='*60}")
print(exai_text)
print(f"\n--- Total length: {len(exai_text)} characters ---")

EXAI Context for: Terfenadine + Alprazolam
## MLP Baseline (Molecular Fingerprints)
- Prediction score: 0.9078
- Shared CYP enzymes: Cytochrome P450 3A4, Cytochrome P450 3A5, Cytochrome P450 3A7
- Shared targets: None
- Drug A (Terfenadine) top fingerprint bits: [1437, 808, 925, 1261, 1610]
- Drug B (Alprazolam) top fingerprint bits: [168, 1571, 825, 982, 843]

## GraphSAGE (topo) - Perturbation Analysis
- Prediction score: 0.8658 (logit: 1.8644)
- Top 5 influential neighbors:
  - Adipiplon (impact: 0.0107, CYP: none)
  - Oxiracetam (impact: 0.0099, CYP: none)
  - Ethchlorvynol (impact: 0.0097, CYP: none)
  - Nomifensine (impact: 0.0097, CYP: none)
  - Meprobamate (impact: 0.0096, CYP: none)

## GraphSAGE (feat) - Perturbation Analysis
- Prediction score: 0.8622 (logit: 1.8336)
- Top 5 influential neighbors:
  - Diacerein (impact: 0.0017, CYP: none)
  - Adipiplon (impact: 0.0017, CYP: none)
  - Rhein (impact: 0.0016, CYP: none)
  - Cabozantinib (impact: 0.0016, CYP: CYP2C9, CYP3A4)
  -

In [ ]:
# ============================================
# STEP 4f: Resolve GAT attention indices to drug names
# ============================================
#
# WHY:  GAT attention top10 stores OGB indices (e.g., 3397) but
#       Claude needs drug names to generate meaningful explanations.
#       We use drug_info to build a quick idx→name lookup.
#
# INPUT:  drug_info DataFrame from Step 1
# OUTPUT: idx_to_name dict, used inside get_exai_context
# ============================================

# Build lookup: ogb_idx → drug name
idx_to_name = dict(zip(drug_info['ogb_idx'].astype(int), drug_info['name']))
print(f"Built lookup for {len(idx_to_name)} drugs")

# Quick test
print(f"Index 3397 → {idx_to_name.get(3397, 'NOT FOUND')}")
print(f"Index 338 → {idx_to_name.get(338, 'NOT FOUND')}")  # should be Alprazolam
print(f"Index 279 → {idx_to_name.get(279, 'NOT FOUND')}")  # should be Terfenadine

Built lookup for 4266 drugs
Index 3397 → Enfortumab vedotin
Index 338 → Alprazolam
Index 279 → Terfenadine


In [ ]:
# ============================================
# STEP 4g: Final EXAI context extractor (with name resolution)
# ============================================
#
# WHAT:  Same as Step 4e but resolves attention neighbor indices
#        to drug names using idx_to_name lookup.
#
# This is the FINAL version we'll use in the RAG pipeline.
# After this, we save checkpoint and move to Step 5 (Claude API).
# ============================================

def get_exai_context(pair_name, mlp_ig, graphsage_perturbation, graphsage_ig,
                     gat_exai, gat_ig, gat_perturbation, idx_to_name):
    """
    Collect all EXAI explanations for a drug pair across all 7 models.
    Returns a formatted string ready to insert into the LLM prompt.
    """

    if pair_name not in VALID_PAIRS:
        return f"No EXAI results available for '{pair_name}'. Only pre-analyzed pairs: {VALID_PAIRS}"

    sections = []

    # ---- 1. MLP: Integrated Gradients on molecular fingerprints ----
    mlp = mlp_ig[pair_name]
    sections.append(f"""## MLP Baseline (Molecular Fingerprints)
- Prediction score: {mlp['prediction']:.4f}
- Shared CYP enzymes: {', '.join(mlp['shared_enzymes']) if mlp['shared_enzymes'] else 'None'}
- Shared targets: {', '.join(mlp['shared_targets']) if mlp['shared_targets'] else 'None'}
- Drug A ({mlp['drug_a']}) top fingerprint bits: {mlp['drug_a_top_bits'][:5]}
- Drug B ({mlp['drug_b']}) top fingerprint bits: {mlp['drug_b_top_bits'][:5]}""")

    # ---- 2. GraphSAGE: Perturbation (topo + feat) ----
    for variant in ['topo', 'feat']:
        gs_data = graphsage_perturbation[variant][pair_name]
        score_prob = sigmoid(gs_data['orig_score'])
        top_neighbors = gs_data['neighbors'][:5]
        neighbor_lines = []
        for n in top_neighbors:
            cyp_str = ', '.join(n['cyp_enzymes']) if n.get('cyp_enzymes') else 'none'
            neighbor_lines.append(f"  - {n['name']} (impact: {n['impact']:.4f}, CYP: {cyp_str})")

        sections.append(f"""## GraphSAGE ({variant}) - Perturbation Analysis
- Prediction score: {score_prob:.4f} (logit: {gs_data['orig_score']:.4f})
- Top 5 influential neighbors:
{chr(10).join(neighbor_lines)}""")

    # ---- 3. GraphSAGE IG (feat model) ----
    gs_ig_pair = graphsage_ig.get(pair_name, {})
    if gs_ig_pair and isinstance(gs_ig_pair, dict):
        for drug_key in ['drug_a', 'drug_b']:
            if drug_key in gs_ig_pair:
                ig_data = gs_ig_pair[drug_key]
                if isinstance(ig_data, dict) and 'top_bits' in ig_data:
                    sections.append(f"""## GraphSAGE (feat) - Integrated Gradients ({drug_key})
- Top fingerprint bits: {ig_data.get('top_bits', [])[:5]}
- Top values: {[f'{v:.4f}' for v in ig_data.get('top_values', [])[:5]]}""")

    # ---- 4. GAT: Attention weights (4 variants) - NOW WITH NAMES ----
    gat_att_pair = gat_exai['attention'].get(pair_name, {})
    for model_name, att_data in gat_att_pair.items():
        if isinstance(att_data, dict):
            top_a = att_data.get('drug_a_top10', [])[:5]
            alphas_a = att_data.get('drug_a_top10_alphas', [])[:5]
            top_b = att_data.get('drug_b_top10', [])[:5]
            alphas_b = att_data.get('drug_b_top10_alphas', [])[:5]

            lines = []
            for idx, alpha in zip(top_a, alphas_a):
                name = idx_to_name.get(int(idx), f"idx_{idx}") if isinstance(idx, (int, float)) else idx
                a_val = f"{alpha:.4f}" if isinstance(alpha, (int, float)) else str(alpha)
                lines.append(f"  - Drug A → {name} (alpha: {a_val})")
            for idx, alpha in zip(top_b, alphas_b):
                name = idx_to_name.get(int(idx), f"idx_{idx}") if isinstance(idx, (int, float)) else idx
                b_val = f"{alpha:.4f}" if isinstance(alpha, (int, float)) else str(alpha)
                lines.append(f"  - Drug B → {name} (alpha: {b_val})")

            sections.append(f"""## {model_name} - Attention Weights
- Top attended neighbors:
{chr(10).join(lines) if lines else '  No attention data available'}""")

    # ---- 5. GAT: Perturbation (4 variants) ----
    gat_pert_pair = gat_perturbation[pair_name]
    for model_name, neighbors in gat_pert_pair.items():
        if isinstance(neighbors, list) and neighbors:
            score_prob = sigmoid(neighbors[0].get('baseline_score', 0))
            top_5 = neighbors[:5]
            neighbor_lines = []
            for n in top_5:
                name = n.get('neighbor_name', n.get('neighbor_idx', '?'))
                delta = n.get('abs_delta', 0)
                neighbor_lines.append(f"  - {name} (abs_delta: {delta:.6f})")

            sections.append(f"""## {model_name} - Perturbation Analysis
- Prediction score: {score_prob:.4f}
- Top 5 influential neighbors:
{chr(10).join(neighbor_lines)}""")

    # ---- 6. GAT: Integrated Gradients (feat models only) ----
    gat_ig_pair = gat_ig.get(pair_name, {})
    for model_name, ig_data in gat_ig_pair.items():
        if isinstance(ig_data, dict):
            top_a_bits = ig_data.get('drug_a_top_bits', [])[:5]
            top_a_vals = ig_data.get('drug_a_top_values', [])[:5]
            top_b_bits = ig_data.get('drug_b_top_bits', [])[:5]
            top_b_vals = ig_data.get('drug_b_top_values', [])[:5]

            sections.append(f"""## {model_name} - Integrated Gradients
- Drug A top bits: {top_a_bits}, values: {[f'{v:.4f}' for v in top_a_vals]}
- Drug B top bits: {top_b_bits}, values: {[f'{v:.4f}' for v in top_b_vals]}""")

    return '\n\n'.join(sections)


# --- FINAL TEST ---
pair = 'Terfenadine + Alprazolam'
exai_text = get_exai_context(pair, mlp_ig, graphsage_perturbation, graphsage_ig,
                              gat_exai, gat_ig, gat_perturbation, idx_to_name)
print(f"EXAI Context for: {pair}")
print(f"{'='*60}")
print(exai_text)

EXAI Context for: Terfenadine + Alprazolam
## MLP Baseline (Molecular Fingerprints)
- Prediction score: 0.9078
- Shared CYP enzymes: Cytochrome P450 3A4, Cytochrome P450 3A5, Cytochrome P450 3A7
- Shared targets: None
- Drug A (Terfenadine) top fingerprint bits: [1437, 808, 925, 1261, 1610]
- Drug B (Alprazolam) top fingerprint bits: [168, 1571, 825, 982, 843]

## GraphSAGE (topo) - Perturbation Analysis
- Prediction score: 0.8658 (logit: 1.8644)
- Top 5 influential neighbors:
  - Adipiplon (impact: 0.0107, CYP: none)
  - Oxiracetam (impact: 0.0099, CYP: none)
  - Ethchlorvynol (impact: 0.0097, CYP: none)
  - Nomifensine (impact: 0.0097, CYP: none)
  - Meprobamate (impact: 0.0096, CYP: none)

## GraphSAGE (feat) - Perturbation Analysis
- Prediction score: 0.8622 (logit: 1.8336)
- Top 5 influential neighbors:
  - Diacerein (impact: 0.0017, CYP: none)
  - Adipiplon (impact: 0.0017, CYP: none)
  - Rhein (impact: 0.0016, CYP: none)
  - Cabozantinib (impact: 0.0016, CYP: CYP2C9, CYP3A4)
  -

In [ ]:
BASE = '/content/drive/MyDrive/ddi_capstone'
SAVE_PATH = f'{BASE}/rag_pipeline'

In [ ]:
# ============================================
# CHECKPOINT: Save all Step 4 artifacts
# ============================================
#
# Saves: idx_to_name lookup + the final exai extractor function
# After restart, restore with: restore_session.py + load EXAI JSONs
# NEXT: Step 5 - Claude API integration
# ============================================

import json
import pickle

# Save idx_to_name lookup
with open(f'{SAVE_PATH}/idx_to_name.json', 'w') as f:
    json.dump({str(k): v for k, v in idx_to_name.items()}, f)

print(f"Saved idx_to_name ({len(idx_to_name)} entries) to {SAVE_PATH}/idx_to_name.json")
print(f"\nAll Step 4 artifacts saved. Ready for Step 5 (Claude API).")

Saved idx_to_name (4266 entries) to /content/drive/MyDrive/ddi_capstone/rag_pipeline/idx_to_name.json

All Step 4 artifacts saved. Ready for Step 5 (Claude API).


In [ ]:
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')
print(f"Key loaded: {api_key[:10]}...{api_key[-4:]}")

Key loaded: sk-ant-api...IAAA


In [ ]:
!pip install -q chromadb sentence-transformers

In [ ]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

BASE = '/content/drive/MyDrive/ddi_capstone'
SAVE_PATH = f'{BASE}/rag_pipeline'

with open(f'{SAVE_PATH}/drug_chunks.json', 'r') as f:
    chunks = json.load(f)

client = chromadb.PersistentClient(path=f'{SAVE_PATH}/chromadb')
collection = client.get_collection('ddi_drugs')

print(f"Restored: {len(chunks)} chunks, {collection.count()} indexed documents")

Restored: 4266 chunks, 4266 indexed documents


In [ ]:
# ============================================
# STEP 5: Claude API Integration
# ============================================
#
# WHAT: Send the combined context (DrugBank retrieval from Step 3 +
#       EXAI explanations from Step 4) to Claude API and get a
#       pharmacologically-informed natural language explanation.
#
# WHY:  This is the core of the RAG system — Claude synthesizes
#       structured model outputs + pharmacological knowledge into
#       an explanation a clinician can understand.
#
# INPUT:  retrieve_drug_context() from Step 3c
#         get_exai_context() from Step 4g
# OUTPUT: Natural language DDI explanation grounded in evidence
# ============================================

!pip install -q anthropic

from google.colab import userdata
import anthropic

# Initialize Claude client (key from Colab Secrets, never hardcoded)
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

SYSTEM_PROMPT = """You are a clinical pharmacology expert explaining drug-drug interactions (DDIs)
predicted by machine learning models.

Rules:
- Only state facts that are directly supported by the retrieved DrugBank/PubChem/DailyMed data
  or the EXAI model outputs provided. If information is missing or unclear, say so explicitly.
- Do not speculate or fill gaps with general pharmacological knowledge not present in the context.
- Do not use emojis, star ratings, or decorative symbols. Use plain text only.
- Use concise paragraphs, not excessive bullet points.
- Reference specific drug names, enzyme names (e.g., CYP3A4), and model outputs by name.

Structure your response with these sections:

## Interaction Summary
Two to three sentences: what interaction is predicted and at what confidence level across models.

## Pharmacological Mechanism
What the retrieved data (DrugBank, PubChem, DailyMed) says about why these drugs interact.
Separate pharmacokinetic (enzyme/transporter) from pharmacodynamic (target/receptor) mechanisms.
If the retrieved data does not document a mechanism, state that explicitly.

## Model Evidence
For each architecture (MLP, GraphSAGE, GAT), one short paragraph summarizing:
- What the EXAI method revealed (IG attributions, perturbation neighbors, attention weights)
- Whether that evidence is biologically coherent based on the retrieved data

## Cross-Model Agreement
Where models agree and where they diverge. Keep this to one paragraph.

## Confidence and Limitations
How well the model explanations align with the retrieved pharmacological data.
Explicitly state what the models fail to capture or where evidence is insufficient.

Target length: 400-600 words. Be direct and precise."""


def generate_ddi_explanation(drug_a, drug_b, pair_name, collection,
                              mlp_ig, graphsage_perturbation, graphsage_ig,
                              gat_exai, gat_ig, gat_perturbation, idx_to_name):
    """
    Full RAG pipeline: retrieve context + get EXAI results + call Claude.
    Returns Claude's explanation.
    """

    # Step A: Retrieve pharmacological context from ChromaDB (Step 3c)
    retrieval = retrieve_drug_context(drug_a, drug_b, collection)

    # Step B: Get EXAI explanations from all models (Step 4g)
    exai_text = get_exai_context(pair_name, mlp_ig, graphsage_perturbation,
                                  graphsage_ig, gat_exai, gat_ig, gat_perturbation,
                                  idx_to_name)

    # Step C: Build the prompt combining both sources
    user_prompt = f"""Analyze the predicted drug-drug interaction between {drug_a} and {drug_b}.

## Retrieved Pharmacological Data (DrugBank)

### {drug_a}
{retrieval['drug_a']['document']}

### {drug_b}
{retrieval['drug_b']['document']}

## Model Explainability (EXAI) Results
The following results come from 7 GNN model variants analyzing this drug pair:

{exai_text}

## Question
Based on the pharmacological data and model explanations above, provide an evidence-based
explanation of why these models predict an interaction between {drug_a} and {drug_b}.
Highlight which models best capture the known pharmacological mechanism."""

    # Step D: Call Claude API
    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2048,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}]
    )

    return {
        'explanation': response.content[0].text,
        'drug_a': drug_a,
        'drug_b': drug_b,
        'model': response.model,
        'usage': {
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens
        }
    }


# --- TEST: Generate explanation for first validation pair ---
print("Generating DDI explanation...")
print("(This calls Claude API — may take a few seconds)\n")

result = generate_ddi_explanation(
    drug_a="Terfenadine",
    drug_b="Alprazolam",
    pair_name="Terfenadine + Alprazolam",
    collection=collection,
    mlp_ig=mlp_ig,
    graphsage_perturbation=graphsage_perturbation,
    graphsage_ig=graphsage_ig,
    gat_exai=gat_exai,
    gat_ig=gat_ig,
    gat_perturbation=gat_perturbation,
    idx_to_name=idx_to_name
)

print(f"Model: {result['model']}")
print(f"Tokens used: {result['usage']['input_tokens']} in / {result['usage']['output_tokens']} out")
print(f"\n{'='*60}")
print(result['explanation'])

Generating DDI explanation...
(This calls Claude API — may take a few seconds)

Model: claude-sonnet-4-5-20250929
Tokens used: 3504 in / 1167 out

## Interaction Summary

All models predict a high-probability interaction between terfenadine and alprazolam, with scores ranging from 0.8123 to 0.9078. The MLP baseline achieves the highest confidence (0.9078), while GNN architectures consistently predict interaction probabilities between 0.81-0.87. This cross-architecture agreement suggests a genuine pharmacological interaction, though the specific mechanisms captured vary substantially by model type.

## Pharmacological Mechanism

The retrieved data documents a clear pharmacokinetic mechanism: both drugs are extensively metabolized by the cytochrome P450 3A family, specifically CYP3A4, CYP3A5, and CYP3A7. Terfenadine is a prodrug that requires near-complete hepatic metabolism by CYP3A4 to form its active metabolite fexofenadine. The data explicitly states terfenadine is "generally complet

In [ ]:
# ============================================
# STEP 5b: Generate and display explanations for all 5 validation pairs
# ============================================
#
# WHAT: Run the full RAG pipeline on all 5 pre-analyzed drug pairs
#       and display results with formatted markdown rendering.
#
# WHY:  We need explanations for all validation pairs to:
#       1. Verify consistency across different interaction types
#       2. Have complete results for the paper/capstone
#       3. Save as JSON for future use in the DDI Explorer UI
#
# INPUT:  All EXAI results + ChromaDB collection + Claude API
# OUTPUT: Formatted explanations displayed in notebook + saved to Drive
# ============================================

import time
from IPython.display import display, Markdown

# Drug pair info: pair_name → (drug_a, drug_b)
PAIR_INFO = {
    'Terfenadine + Alprazolam': ('Terfenadine', 'Alprazolam'),
    'Nilotinib + Dacomitinib': ('Nilotinib', 'Dacomitinib'),
    'Palonosetron + Clomipramine': ('Palonosetron', 'Clomipramine'),
    'Flunitrazepam + Alprazolam': ('Flunitrazepam', 'Alprazolam'),
    'Clomipramine + Atomoxetine': ('Clomipramine', 'Atomoxetine')
}

all_results = {}

for pair_name, (drug_a, drug_b) in PAIR_INFO.items():
    print(f"Generating: {pair_name}...")

    try:
        result = generate_ddi_explanation(
            drug_a=drug_a,
            drug_b=drug_b,
            pair_name=pair_name,
            collection=collection,
            mlp_ig=mlp_ig,
            graphsage_perturbation=graphsage_perturbation,
            graphsage_ig=graphsage_ig,
            gat_exai=gat_exai,
            gat_ig=gat_ig,
            gat_perturbation=gat_perturbation,
            idx_to_name=idx_to_name
        )

        all_results[pair_name] = result

        # Display with nice formatting
        tokens_in = result['usage']['input_tokens']
        tokens_out = result['usage']['output_tokens']
        display(Markdown(f"""---
###  {pair_name}
*Model: {result['model']} | Tokens: {tokens_in} in / {tokens_out} out*

{result['explanation']}
"""))

        # Small delay to avoid rate limits
        time.sleep(2)

    except Exception as e:
        print(f"ERROR on {pair_name}: {e}")
        all_results[pair_name] = {'error': str(e)}

# --- SUMMARY ---
total_in = sum(r['usage']['input_tokens'] for r in all_results.values() if 'usage' in r)
total_out = sum(r['usage']['output_tokens'] for r in all_results.values() if 'usage' in r)
cost = (total_in * 3 + total_out * 15) / 1_000_000

display(Markdown(f"""---
## Summary
| Metric | Value |
|--------|-------|
| Pairs processed | {sum(1 for r in all_results.values() if 'explanation' in r)}/5 |
| Total input tokens | {total_in:,} |
| Total output tokens | {total_out:,} |
| Estimated cost | ${cost:.4f} |
"""))

Generating: Terfenadine + Alprazolam...


---
###  Terfenadine + Alprazolam
*Model: claude-sonnet-4-5-20250929 | Tokens: 3504 in / 1130 out*

# Evidence-Based Analysis: Terfenadine-Alprazolam Interaction

## Interaction Summary

All seven model variants predict a clinically significant interaction between terfenadine and alprazolam with high confidence (scores ranging from 0.8123 to 0.9078). The MLP baseline shows the strongest prediction at 0.9078, while graph-based models converge around 0.84-0.86. This consistency across architectures suggests a robust signal, though the models detect this interaction through different mechanistic pathways than those documented in the retrieved pharmacological data.

## Pharmacological Mechanism

The retrieved DrugBank data documents that both terfenadine and alprazolam are extensively metabolized by the CYP3A family, specifically sharing CYP3A4, CYP3A5, and CYP3A7 as metabolic enzymes. Terfenadine is explicitly described as a prodrug that undergoes near-complete first-pass metabolism by CYP3A4 to its active metabolite fexofenadine. Alprazolam undergoes hepatic hydroxylation to alpha-hydroxyalprazolam, also active, with metabolism mediated by CYP3A4 among other enzymes.

This creates a clear pharmacokinetic basis for interaction: competitive inhibition at CYP3A4 could impair terfenadine's conversion to fexofenadine, leading to accumulation of parent terfenadine, which has documented cardiotoxicity via KCNH2 (hERG) channel blockade. The retrieved data does not document any shared pharmacodynamic targets—terfenadine acts on histamine H1 receptors and muscarinic receptors, while alprazolam modulates GABA-A receptors. No direct pharmacodynamic mechanism is evident from the retrieved information.

## Model Evidence

The MLP baseline correctly identifies the shared CYP3A enzymes and achieves the highest prediction score. However, its top fingerprint bits for each drug show no overlap, suggesting the model learns interaction patterns indirectly through structural features correlated with CYP metabolism rather than explicitly encoding the enzyme overlap.

GraphSAGE models, both topology-based (0.8658) and feature-based (0.8622), identify influential neighbors that lack documented CYP3A involvement. The topology variant highlights CNS depressants (adipiplon, meprobamate, ethchlorvynol) while the feature variant attends to diacerein and cabozantinib. Only cabozantinib shares CYP3A4 metabolism. These perturbation analyses suggest the models learned association patterns from CNS drug classes rather than capturing the specific CYP-mediated mechanism.

GAT architectures show highly variable attention patterns across configurations. GAT_base_feat attends to warfarin and anticoagulants (attention weights ~0.0025), which are CYP2C9 substrates but not mechanistically relevant here. GAT_skip_feat focuses on fluoroquinolones (pefloxacin, gatifloxacin), some of which are CYP3A substrates but represent a different therapeutic class. Perturbation analyses from GAT variants identify corticosteroids (clobetasol, methylprednisolone) and other CYP3A substrates, showing partial alignment with the pharmacokinetic mechanism. Integrated gradient attributions reveal specific molecular fingerprint regions but without clear interpretability linking to CYP metabolism.

## Cross-Model Agreement

All models converge on high interaction probability, demonstrating robust prediction. However, they diverge substantially in their explanatory pathways. The MLP leverages molecular structure patterns, GraphSAGE models rely on neighborhood similarity to CNS depressants, and GAT variants attend to diverse CYP3A substrate classes (warfarin derivatives, fluoroquinolones, corticosteroids) without consistent mechanistic focus. No model explicitly surfaces the shared CYP3A4/3A5/3A7 metabolism as the primary interaction driver, despite this being the documented pharmacological basis.

## Confidence and Limitations

The models successfully predict interaction risk but fail to transparently capture the known CYP3A-mediated mechanism. The explanations reveal that models learn interaction patterns through proxy signals—therapeutic class similarity, structural motifs, and neighborhood associations—rather than explicit enzyme-level reasoning. The retrieved pharmacological data clearly documents shared CYP3A metabolism, yet this critical feature remains implicit in model representations. Perturbation neighbors and attention weights predominantly identify drugs from overlapping therapeutic classes or metabolic pathways without pinpointing the specific CYP3A4 competition mechanism. This gap indicates that while the models achieve accurate risk stratification, their explanations lack the pharmacological specificity needed for clinical decision support. Clinicians should recognize that the model's high confidence reflects learned correlations rather than mechanistic understanding of the terfenadine cardiotoxicity risk when CYP3A4 is competitively inhibited.


Generating: Nilotinib + Dacomitinib...


---
###  Nilotinib + Dacomitinib
*Model: claude-sonnet-4-5-20250929 | Tokens: 3534 in / 1239 out*

## Interaction Summary

All seven model architectures predict a clinically significant interaction between nilotinib and dacomitinib with prediction scores ranging from 0.7787 to 0.9084. The MLP baseline achieved the highest confidence (0.9084), while feature-augmented graph models showed slightly lower but still strong predictions (0.78-0.85). This cross-model consensus suggests a robust interaction signal driven by overlapping metabolic pathways.

## Pharmacological Mechanism

The retrieved DrugBank data documents extensive overlap in cytochrome P450 metabolism between these drugs. Both nilotinib and dacomitinib are substrates of CYP3A4, CYP2D6, and CYP2C9. Nilotinib is extensively metabolized principally by CYP3A4, where it represents the principal circulating component with metabolites contributing minimally to pharmacologic activity. Dacomitinib undergoes oxidative metabolism primarily via CYP2D6 (forming the major metabolite PF-05199265) and CYP2C9, with subsequent metabolism by CYP3A4 for smaller metabolites.

Both drugs are also substrates of the efflux transporters ABCB1 (P-glycoprotein) and ABCG2, which could contribute to pharmacokinetic interactions at the absorption and distribution level. Additionally, both undergo UGT1A1-mediated glucuronidation, creating another potential point of metabolic competition.

No pharmacodynamic mechanism is documented in the retrieved data. The drugs target different receptor families (nilotinib: BCR-ABL, PDGFR, c-kit; dacomitinib: EGFR/HER family) with no shared therapeutic targets, indicating the interaction is purely pharmacokinetic.

## Model Evidence

The MLP model achieved 0.9084 prediction confidence and explicitly identified the three shared CYP enzymes (3A4, 2D6, 2C9) as interaction determinants. Its high performance despite using only molecular fingerprints suggests structural features encoding metabolic liability are sufficient for prediction, though the non-overlapping top fingerprint bits between drugs (nilotinib: 443, 219, 1795; dacomitinib: 1157, 673, 465) indicate the model captures complementary rather than identical structural motifs.

GraphSAGE models (topology and feature-based) showed 0.8468-0.8473 predictions. The topology variant identified perturbation neighbors with no shared CYP enzymes (ilaprazole, darbepoetin alfa), suggesting reliance on network position rather than metabolic evidence. The feature-based variant performed marginally better by identifying avapritinib (CYP2C9, 3A4 substrate) and butyrfentanyl (CYP2D6, 3A4 substrate) as influential neighbors, partially recovering metabolic coherence.

GAT architectures showed variable attention patterns. GAT_base_feat attended to warfarin compounds and bile acids, which share CYP2C9 metabolism but lack direct pharmacological relevance to tyrosine kinase inhibitors. GAT_skip_feat focused on fluoroquinolones (moxifloxacin, pefloxacin), which do not share the relevant CYP pathways. Perturbation analysis across GAT models identified diverse neighbors (omadacycline, hydroflumethiazide, gatifloxacin) with minimal CYP overlap, indicating these models struggle to ground predictions in the documented metabolic mechanism.

Integrated gradient analysis on GAT_skip_feat revealed non-overlapping important fingerprint bits between the two drugs, similar to the MLP pattern, but failed to highlight metabolic features explicitly.

## Cross-Model Agreement

All models converged on a strong interaction prediction (0.78-0.91 range), but diverged substantially in mechanistic interpretation. The MLP baseline uniquely surfaced the shared CYP enzyme evidence that directly matches DrugBank documentation. Graph-based models achieved comparable predictive performance but relied on network neighbors with weak or absent metabolic connections to the target drug pair. Feature-augmented variants (GraphSAGE_feat, GAT_skip_feat) showed marginal improvement over topology-only versions but did not consistently recover the CYP3A4/2D6/2C9 overlap.

## Confidence and Limitations

The MLP model's explicit identification of shared CYP enzymes provides the strongest alignment with retrieved pharmacological data, offering a mechanistically interpretable explanation grounded in competitive inhibition or shared metabolic burden. Graph neural network architectures failed to leverage their structural advantages effectively—attention weights and perturbation neighbors rarely pointed to drugs sharing the documented CYP pathways. The models collectively miss the transporter overlap (ABCB1, ABCG2, UGT1A1) that could compound pharmacokinetic interactions. None of the EXAI outputs reference the specific metabolic sequence documented for dacomitinib (CYP2D6 → CYP3A4) or nilotinib's predominant CYP3A4 dependence, suggesting models capture interaction risk without resolving detailed mechanistic pathways. The high prediction confidence across architectures likely reflects successful pattern recognition of co-prescribed tyrosine kinase inhibitors with overlapping metabolism, but explanatory coherence remains limited to the MLP's enzyme-level features.


Generating: Palonosetron + Clomipramine...


...

Generating: Flunitrazepam + Alprazolam...


---
###  Flunitrazepam + Alprazolam
*Model: claude-sonnet-4-5-20250929 | Tokens: 3489 in / 1296 out*

## Interaction Summary

All models predict a strong interaction between flunitrazepam and alprazolam, with prediction scores ranging from 0.8252 to 0.9077. This high confidence is pharmacologically justified: both drugs are benzodiazepines that share identical molecular targets (GABA(A) receptor and its benzodiazepine binding site) and overlapping metabolic pathways (CYP3A4 and CYP2C9). The interaction represents both pharmacodynamic synergy at GABA receptors and potential pharmacokinetic competition for hepatic metabolism.

## Pharmacological Mechanism

**Pharmacodynamic mechanism:** DrugBank and PubChem data confirm both drugs bind to the same GABA(A) receptor benzodiazepine binding site and translocator protein. Both enhance GABAergic inhibitory neurotransmission, producing sedation, muscle relaxation, and anxiolysis through the same molecular targets. Concurrent administration would amplify central nervous system depression through additive or synergistic GABA receptor modulation.

**Pharmacokinetic mechanism:** Both benzodiazepines undergo hepatic metabolism. Flunitrazepam is metabolized by CYP3A4, CYP2C9, CYP2C19, CYP2E1, CYP2A6, and CYP2B6, plus UGT enzymes. Alprazolam is primarily metabolized by CYP3A4 and CYP3A5, with minor CYP2C9 involvement, undergoing hydroxylation to alpha-hydroxyalprazolam. The shared CYP3A4 and CYP2C9 pathways create potential for competitive inhibition, though the clinical significance depends on substrate affinity and dose. Flunitrazepam has an 18-26 hour half-life versus alprazolam's 6.3-26.9 hours, suggesting different duration profiles that could complicate combined use.

## Model Evidence

**MLP Baseline:** The 0.9077 prediction score correctly identifies shared CYP2C9 and CYP3A4 enzymes and identical GABA(A) receptor targets. The molecular fingerprint analysis captured structural features distinguishing these benzodiazepines (different top bits for each drug), suggesting the model recognizes both similarity and subtle structural differences. However, fingerprints alone cannot distinguish pharmacodynamic from pharmacokinetic contributions.

**GraphSAGE models:** Both topology-based (0.8690) and feature-based (0.8642) variants identified structurally similar GABA-modulating compounds as influential neighbors (adipiplon, pagoclone, gedocarnil). These neighbors lack documented CYP interactions but share GABAergic mechanisms, indicating GraphSAGE emphasizes pharmacodynamic similarity in the learned graph structure. The perturbation analysis reveals the model relies on neighborhood context of sedative-hypnotic agents rather than explicit metabolic pathway overlap.

**GAT architectures:** Attention patterns varied dramatically across variants. GAT_base_feat attended to warfarin compounds (metabolized by CYP2C9), ethinylestradiol, and CYP3A4 substrates like dexamethasone and betamethasone in skip_feat variant, suggesting these models learned to focus on metabolic pathway competition. The GAT_skip model's attention to yohimbine and phenylephrine (adrenergic agents) appears pharmacologically irrelevant. Perturbation analysis shows GAT_skip_feat identifying benzodiazepines (halazepam, zolazepam, cinolazepam) as influential, which aligns with pharmacodynamic class effects. Integrated Gradients in GAT_skip_feat revealed overlapping important fingerprint bits (135 for both drugs, 470 and 724 appearing in both), indicating structural commonalities the model uses for prediction.

## Cross-Model Agreement

All models achieve 0.82-0.91 prediction scores, demonstrating strong consensus. GraphSAGE models converge around 0.86-0.87, while MLP achieves highest confidence at 0.91. The primary divergence lies in explanation focus: MLP and feature-enriched GAT variants emphasize metabolic enzyme overlap (CYP3A4/2C9), while topology-based GraphSAGE and certain GAT variants prioritize pharmacodynamic class membership through GABAergic neighbor compounds. This suggests multiple valid explanation pathways converge on the same high-risk prediction.

## Confidence and Limitations

The models correctly capture the dual interaction mechanism: shared GABA(A) receptor targets (pharmacodynamic) and overlapping CYP-mediated metabolism (pharmacokinetic). Feature-enriched models (MLP, GAT_base_feat, GAT_skip_feat) best align with retrieved data by identifying specific enzyme overlap. However, no model quantifies the relative contribution of pharmacodynamic synergy versus metabolic competition—the retrieved data suggests the former dominates clinically, as both are CNS depressants with documented additive sedation risk. 

GraphSAGE perturbation neighbors (adipiplon, pagoclone) lack DrugBank CYP documentation, indicating the model learned GABA-modulating drug clusters without explicit metabolic annotations. GAT attention to unrelated drug classes (warfarin, phenylephrine) suggests some architectural variants learn spurious correlations. The models fail to distinguish that CYP3A4 competition may be clinically minor compared to the well-established danger of combined benzodiazepine CNS depression, which the pharmacological data emphasizes through warnings about respiratory depression and sedation.


Generating: Clomipramine + Atomoxetine...


---
###  Clomipramine + Atomoxetine
*Model: claude-sonnet-4-5-20250929 | Tokens: 3508 in / 1216 out*

## Interaction Summary

All models predict a high-probability drug-drug interaction between clomipramine and atomoxetine, with prediction scores ranging from 0.8383 to 0.9079. The MLP baseline achieves the highest confidence at 0.9079. This interaction is strongly supported by documented pharmacological overlap: both drugs are metabolized by CYP2D6 and CYP2C19, and both inhibit the noradrenaline transporter and serotonin transporter, creating risks for both pharmacokinetic and pharmacodynamic interactions.

## Pharmacological Mechanism

The retrieved data documents two distinct interaction mechanisms. Pharmacokinetically, clomipramine undergoes extensive hepatic metabolism via CYP2C19, CYP3A4, and CYP1A2, with N-demethylation producing the active metabolite desmethylclomipramine. Atomoxetine is primarily metabolized by CYP2D6 to 4-hydroxyatomoxetine. Both drugs share CYP2D6 and CYP2C19 pathways, creating potential for competitive inhibition that could elevate plasma concentrations of either drug.

Pharmacodynamically, clomipramine inhibits both the serotonin transporter and noradrenaline transporter, while atomoxetine is a selective norepinephrine reuptake inhibitor that also affects the serotonin transporter. The retrieved data shows clomipramine targets sodium-dependent serotonin transporter, 5-HT2A/2B/2C receptors, and sodium-dependent noradrenaline transporter. Atomoxetine targets sodium-dependent noradrenaline transporter as its primary site, with secondary effects on the serotonin transporter. Co-administration could produce additive inhibition of norepinephrine reuptake, potentially causing cardiovascular effects, as the atomoxetine data explicitly notes risks of increased blood pressure, tachycardia, and in severe cases, sudden death or myocardial infarction.

## Model Evidence

The MLP baseline correctly identifies the shared CYP2C19 and CYP2D6 enzymes and shared transporter targets (noradrenaline and serotonin transporters). This model directly captures the documented interaction mechanisms through explicit feature encoding. However, the specific fingerprint bits provide limited interpretability regarding which molecular substructures drive the prediction.

GraphSAGE models achieve lower scores (0.8633-0.8712) and identify pharmacologically irrelevant neighbors. The topology-based variant highlights auranofin, octinoxate, and contrast agents with no CYP overlap, while the feature-based variant emphasizes imaging agents like gadofosveset and iodixanol. Neither GraphSAGE variant captures neighbors that would explain the CYP2D6/2C19 or transporter-mediated mechanisms documented in the retrieved data.

GAT models show substantial variation (0.8383-0.8543) with attention distributed across structurally dissimilar drugs. GAT_skip attends to letermovir (a CYP3A4 inhibitor) with high weight (0.1339), but letermovir does not share the CYP2D6 pathway critical to this interaction. GAT_base_feat and GAT_skip_feat focus on cephalosporin antibiotics, which lack documented relevance to tricyclic antidepressant or norepinephrine reuptake inhibitor mechanisms. Perturbation analysis identifies letermovir and teneligliptin as influential for GAT_skip, but these neighbors do not explain the shared transporter targets. The integrated gradients show high attributions to fingerprint bits in GAT_skip_feat, but without mapping these bits to specific substructures, their biological meaning remains unclear.

## Cross-Model Agreement

All models converge on a high-risk interaction prediction, with scores clustered between 0.84 and 0.91. However, models diverge substantially in their explanatory pathways. The MLP baseline alone explicitly captures the shared enzymatic and target features documented in DrugBank. Graph-based models identify chemically and pharmacologically dissimilar neighbors, suggesting they rely on latent network patterns rather than direct pharmacological similarity to make predictions.

## Confidence and Limitations

The MLP model's explanations align precisely with retrieved pharmacological data, correctly identifying CYP2D6, CYP2C19, and dual transporter inhibition as interaction drivers. In contrast, GNN architectures fail to surface pharmacologically coherent neighbors or attention patterns. GraphSAGE and GAT models identify contrast agents, antibiotics, and unrelated CNS drugs that do not share the metabolic or target profiles documented for clomipramine and atomoxetine. This suggests GNN models may learn interaction prediction from higher-order topological patterns not captured in the retrieved enzyme and target annotations. The models do not address the documented cardiovascular risks of combined norepinephrine reuptake inhibition. No model explanations reference clomipramine's 5-HT2A receptor activity or atomoxetine's effects on NMDA receptors, indicating incomplete mechanism coverage. The high prediction confidence likely reflects the strong CYP and transporter overlap, but the GNN explanations provide limited clinical insight beyond what the MLP baseline captures through direct feature matching.


---
## Summary
| Metric | Value |
|--------|-------|
| Pairs processed | 5/5 |
| Total input tokens | 17,594 |
| Total output tokens | 6,063 |
| Estimated cost | $0.1437 |


In [ ]:
# Chequeá qué variable existe
for name in ['enriched_results', 'all_results', 'results']:
    if name in dir():
        print(f"Found: {name}")

Found: all_results
Found: results


In [ ]:
with open(f'{SAVE_PATH}/rag_explanations_enriched.json', 'w') as f:
    json.dump(all_results, f, indent=2)

with open(f'{SAVE_PATH}/rag_explanations_enriched.md', 'w') as f:
    for pair_name, result in all_results.items():
        if 'explanation' in result:
            f.write(f"# {pair_name}\n\n")
            f.write(result['explanation'])
            f.write(f"\n\n---\n\n")

print("Saved")

Saved


In [ ]:
# ============================================
# CHECKPOINT: Save all RAG explanations to Drive
# ============================================
#
# Saves two formats:
#   1. JSON (structured, for DDI Explorer integration in Step 6)
#   2. Markdown (readable, for paper/presentation)
#
# TO RESTORE: json.load('rag_pipeline/rag_explanations.json')
# ============================================

import json

# Save full results with metadata
with open(f'{SAVE_PATH}/rag_explanations.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Save readable markdown version
with open(f'{SAVE_PATH}/rag_explanations.md', 'w') as f:
    for pair_name, result in all_results.items():
        if 'explanation' in result:
            f.write(f"# {pair_name}\n\n")
            f.write(result['explanation'])
            f.write(f"\n\n---\n\n")

print(f"Saved {len(all_results)} explanations to:")
print(f"  - {SAVE_PATH}/rag_explanations.json")
print(f"  - {SAVE_PATH}/rag_explanations.md")

## PubChem

In [ ]:
# ============================================
# STEP 7: Enrich Knowledge Base with PubChem data
# ============================================
#
# WHAT: Query PubChem REST API to get pharmacological descriptions,
#       mechanism of action, and pharmacology text for each drug.
#       This adds the narrative context that DrugBank CSVs lack.
#
# WHY:  Our current chunks (from Step 2) have structured data
#       (enzymes, targets, transporters) but no free-text descriptions.
#       PubChem provides detailed pharmacology text that will give
#       Claude richer context to generate explanations.
#
# HOW:  We already have PubChem CIDs in drug_mapping_smiles.csv
#       from our molecular data. We use those CIDs to query
#       PubChem's PUG REST API — no API key needed.
#
# INPUT:  drug_mapping_smiles.csv (node_idx → pubchem_cid mapping)
#         drug_info DataFrame from Step 1
# OUTPUT: Enriched chunks with PubChem pharmacology text, saved to Drive
# ============================================

import requests
import time
import json

# First, check our PubChem CID coverage
drug_mapping = pd.read_csv(f'{BASE}/molecular/data/drug_mapping_smiles.csv')
print("drug_mapping columns:", drug_mapping.columns.tolist())
print(f"Total drugs: {len(drug_mapping)}")
print(f"Drugs with PubChem CID: {drug_mapping['pubchem_cid'].notna().sum()}")
print(f"Coverage: {drug_mapping['pubchem_cid'].notna().sum() / len(drug_mapping) * 100:.1f}%")
print()

# Check mapping for our 5 validation drugs
validation_drugs = ['Terfenadine', 'Alprazolam', 'Nilotinib', 'Dacomitinib',
                    'Palonosetron', 'Clomipramine', 'Flunitrazepam', 'Atomoxetine']

# We need to join drug_mapping (has CID) with drug_info (has names)
# drug_mapping uses 'node idx' and 'drug id', drug_info uses 'ogb_idx' and 'drugbank_id'
mapping_with_names = drug_mapping.merge(
    drug_info[['ogb_idx', 'name']],
    left_on='node idx',
    right_on='ogb_idx',
    how='left'
)

print("Validation drug CIDs:")
for drug in validation_drugs:
    row = mapping_with_names[mapping_with_names['name'] == drug]
    if not row.empty:
        cid = row['pubchem_cid'].values[0]
        print(f"  {drug}: CID = {cid if pd.notna(cid) else 'MISSING'}")
    else:
        print(f"  {drug}: NOT FOUND in mapping")

In [ ]:
# ============================================
# STEP 7b: Query PubChem API for pharmacology descriptions
# ============================================
#
# WHAT: For each drug with a PubChem CID, fetch the pharmacology
#       and mechanism of action text from PubChem's PUG REST API.
#
# WHY:  This free-text pharmacology data is what our DrugBank chunks
#       are missing. It gives Claude narrative context like
#       "Atorvastatin competitively inhibits HMG-CoA reductase..."
#       instead of just structured enzyme/target lists.
#
# HOW:  PubChem PUG REST API at pugrest.ncbi.nlm.nih.gov
#       No API key needed, but rate limited to ~5 requests/second.
#       We query the "pharmacology" and "mechanism_of_action" sections
#       from the compound record.
#
# INPUT:  mapping_with_names DataFrame with PubChem CIDs
# OUTPUT: Dict of CID → pharmacology text, saved to Drive
# ============================================

def fetch_pubchem_pharmacology(cid):
    """
    Fetch pharmacology description and mechanism of action from PubChem.
    Returns a dict with available text fields, or None on failure.
    """
    cid = int(cid)
    base_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON"

    try:
        resp = requests.get(base_url, timeout=15)
        if resp.status_code != 200:
            return None

        data = resp.json()
        sections = data.get('Record', {}).get('Section', [])

        result = {}

        # Walk through sections looking for pharmacology-related content
        for section in sections:
            section_name = section.get('TOCHeading', '')

            if section_name in ['Pharmacology and Biochemistry', 'Drug and Medication Information']:
                for subsection in section.get('Section', []):
                    sub_name = subsection.get('TOCHeading', '')

                    if sub_name in ['Pharmacodynamics', 'Mechanism of Action',
                                     'Absorption, Distribution and Excretion',
                                     'Metabolism/Metabolites']:
                        # Extract text from Information blocks
                        for info in subsection.get('Information', []):
                            value = info.get('Value', {})
                            strings = value.get('StringWithMarkup', [])
                            if strings:
                                text = ' '.join([s.get('String', '') for s in strings])
                                if text and len(text) > 20:
                                    result[sub_name] = text[:1000]  # cap length

        return result if result else None

    except Exception as e:
        return None


# Test with one validation drug first
print("Testing PubChem API with Alprazolam (CID 2118)...")
test = fetch_pubchem_pharmacology(2118)
if test:
    for key, val in test.items():
        print(f"\n--- {key} ---")
        print(val[:300] + "..." if len(val) > 300 else val)
else:
    print("No pharmacology data returned")

In [ ]:
# ============================================
# STEP 7c: Batch fetch PubChem data for all drugs with CIDs
# ============================================
#
# WHAT: Query PubChem for all 3,420 drugs that have CIDs.
#       We start with validation drugs, then do the full batch.
#
# WHY:  Having pharmacology text for all drugs means our RAG system
#       can provide richer context for any drug pair, not just
#       the 5 validation pairs.
#
# NOTE: PubChem rate limits to ~5 req/sec. For 3,420 drugs at
#       0.2s delay per request, this takes ~12 minutes on CPU.
#       We save progress every 100 drugs so Colab restarts
#       dont lose work.
#
# INPUT:  mapping_with_names DataFrame with CIDs
# OUTPUT: pubchem_data.json saved to Drive (incremental checkpoints)
# ============================================

import os

PUBCHEM_SAVE = f'{SAVE_PATH}/pubchem_data.json'

# Load existing progress if any (resume after Colab restart)
if os.path.exists(PUBCHEM_SAVE):
    with open(PUBCHEM_SAVE, 'r') as f:
        pubchem_data = json.load(f)
    print(f"Resuming: {len(pubchem_data)} drugs already fetched")
else:
    pubchem_data = {}
    print("Starting fresh PubChem fetch")

# Build list of drugs to fetch: those with CIDs that we havent fetched yet
drugs_to_fetch = mapping_with_names[mapping_with_names['pubchem_cid'].notna()].copy()
drugs_to_fetch['pubchem_cid'] = drugs_to_fetch['pubchem_cid'].astype(int)

# Filter out already fetched
remaining = drugs_to_fetch[~drugs_to_fetch['pubchem_cid'].astype(str).isin(pubchem_data.keys())]
print(f"Drugs to fetch: {len(remaining)} (of {len(drugs_to_fetch)} total with CIDs)")

# Fetch with progress tracking and incremental saves
fetched = 0
errors = 0
start_time = time.time()

for i, (_, row) in enumerate(remaining.iterrows()):
    cid = int(row['pubchem_cid'])
    name = row.get('name', '')

    result = fetch_pubchem_pharmacology(cid)

    if result:
        pubchem_data[str(cid)] = {
            'name': name,
            'drugbank_id': row['drug id'],
            'data': result
        }
        fetched += 1
    else:
        errors += 1

    # Rate limiting: 0.2s between requests
    time.sleep(0.2)

    # Progress update every 50 drugs
    if (i + 1) % 50 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining_time = (len(remaining) - i - 1) / rate / 60
        print(f"  Progress: {i+1}/{len(remaining)} | "
              f"Fetched: {fetched} | Errors: {errors} | "
              f"~{remaining_time:.1f} min remaining")

    # Save checkpoint every 100 drugs
    if (i + 1) % 100 == 0:
        with open(PUBCHEM_SAVE, 'w') as f:
            json.dump(pubchem_data, f)

# Final save
with open(PUBCHEM_SAVE, 'w') as f:
    json.dump(pubchem_data, f, indent=2)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Total drugs with PubChem data: {len(pubchem_data)}")
print(f"Fetched this run: {fetched}")
print(f"Errors (no pharmacology data): {errors}")

# Quick check: validation drugs
print("\nValidation drug coverage:")
for drug in validation_drugs:
    row = mapping_with_names[mapping_with_names['name'] == drug]
    if not row.empty:
        cid = str(int(row['pubchem_cid'].values[0]))
        has_data = cid in pubchem_data
        sections = list(pubchem_data[cid]['data'].keys()) if has_data else []
        print(f"  {drug}: {'YES' if has_data else 'NO'} - {sections}")

In [ ]:
# ============================================
# STEP 7c-v2: Faster PubChem fetch with parallel requests
# ============================================
#
# WHAT: Same as Step 7c but uses concurrent requests to speed up
#       the fetch from ~83 minutes to ~15-20 minutes.
#
# WHY:  PubChem allows ~5 requests/second. Our previous version
#       did 1 request every 0.2s (5/sec) but the bottleneck was
#       waiting for each response sequentially. With ThreadPool
#       we can have multiple requests in flight at once.
#
# INPUT:  mapping_with_names DataFrame, existing pubchem_data checkpoint
# OUTPUT: pubchem_data.json saved to Drive
# ============================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import os

PUBCHEM_SAVE = f'{SAVE_PATH}/pubchem_data.json'

# Load existing progress
if os.path.exists(PUBCHEM_SAVE):
    with open(PUBCHEM_SAVE, 'r') as f:
        pubchem_data = json.load(f)
    print(f"Resuming: {len(pubchem_data)} drugs already fetched")
else:
    pubchem_data = {}
    print("Starting fresh PubChem fetch")

# Build list of remaining drugs to fetch
drugs_to_fetch = mapping_with_names[mapping_with_names['pubchem_cid'].notna()].copy()
drugs_to_fetch['pubchem_cid'] = drugs_to_fetch['pubchem_cid'].astype(int)
remaining = drugs_to_fetch[~drugs_to_fetch['pubchem_cid'].astype(str).isin(pubchem_data.keys())]
print(f"Drugs to fetch: {len(remaining)} (of {len(drugs_to_fetch)} total with CIDs)")

# Prepare rows as list of dicts for the thread pool
rows = [row.to_dict() for _, row in remaining.iterrows()]

def fetch_one(row):
    """Fetch PubChem data for a single drug. Returns (cid_str, result_dict) or (cid_str, None)."""
    cid = int(row['pubchem_cid'])
    result = fetch_pubchem_pharmacology(cid)
    if result:
        return (str(cid), {
            'name': row.get('name', ''),
            'drugbank_id': row['drug id'],
            'data': result
        })
    return (str(cid), None)

# Run with 5 concurrent threads (respects PubChem rate limit)
fetched = 0
errors = 0
start_time = time.time()
BATCH_SAVE = 200  # save checkpoint every 200 drugs

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_one, row): i for i, row in enumerate(rows)}

    for future in as_completed(futures):
        idx = futures[future]
        cid_str, result = future.result()

        if result:
            pubchem_data[cid_str] = result
            fetched += 1
        else:
            errors += 1

        total_done = fetched + errors

        # Progress update every 100 drugs
        if total_done % 100 == 0:
            elapsed = time.time() - start_time
            rate = total_done / elapsed
            remaining_count = len(rows) - total_done
            remaining_time = remaining_count / rate / 60 if rate > 0 else 0
            print(f"  Progress: {total_done}/{len(rows)} | "
                  f"Fetched: {fetched} | Errors: {errors} | "
                  f"~{remaining_time:.1f} min remaining")

        # Save checkpoint periodically
        if total_done % BATCH_SAVE == 0:
            with open(PUBCHEM_SAVE, 'w') as f:
                json.dump(pubchem_data, f)

# Final save
with open(PUBCHEM_SAVE, 'w') as f:
    json.dump(pubchem_data, f, indent=2)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Total drugs with PubChem data: {len(pubchem_data)}")
print(f"Fetched this run: {fetched}")
print(f"Errors (no pharmacology data): {errors}")

# Validation drug check
print("\nValidation drug coverage:")
for drug in validation_drugs:
    row = mapping_with_names[mapping_with_names['name'] == drug]
    if not row.empty:
        cid = str(int(row['pubchem_cid'].values[0]))
        has_data = cid in pubchem_data
        sections = list(pubchem_data[cid]['data'].keys()) if has_data else []
        print(f"  {drug}: {'YES' if has_data else 'NO'} - {sections}")

In [ ]:
# ============================================
# STEP 8: Fetch DailyMed FDA label data
# ============================================
#
# WHAT: Query the DailyMed API to get official FDA drug label
#       sections, specifically "Drug Interactions" and "Clinical
#       Pharmacology" for each drug.
#
# WHY:  DailyMed provides the FDA-approved label text, which is
#       the gold standard for documented drug interactions.
#       This complements DrugBank (structured data) and PubChem
#       (pharmacology descriptions) with official clinical guidance.
#       Having FDA-documented interactions lets Claude compare
#       model predictions against regulatory evidence.
#
# HOW:  DailyMed REST API at dailymed.nlm.nih.gov/dailymed/services
#       No API key needed. We search by drug name, get the SPL
#       (Structured Product Label), and extract relevant sections.
#
# INPUT:  drug_info DataFrame with drug names
# OUTPUT: dailymed_data.json saved to Drive (incremental checkpoints)
#
# NOTE:  Not all drugs have FDA labels (some are non-US or withdrawn).
#        We expect ~50-60% coverage for our dataset.
# ============================================

import requests
import time
import json
import os
import re

def fetch_dailymed_data(drug_name):
    """
    Fetch drug interaction and clinical pharmacology sections
    from DailyMed FDA labels.

    Steps:
      1. Search DailyMed by drug name to get the SPL set_id
      2. Fetch the full label using that set_id
      3. Extract 'Drug Interactions' and 'Clinical Pharmacology' sections

    Returns dict with available sections, or None on failure.
    """

    # Step 1: Search for the drug by name
    search_url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    params = {'drug_name': drug_name, 'page': 1, 'pagesize': 1}

    try:
        resp = requests.get(search_url, params=params, timeout=15)
        if resp.status_code != 200:
            return None

        search_data = resp.json()
        results = search_data.get('data', [])

        if not results:
            return None

        set_id = results[0].get('setid')
        if not set_id:
            return None

        # Step 2: Fetch the full label sections
        label_url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.json"
        resp = requests.get(label_url, timeout=15)
        if resp.status_code != 200:
            return None

        label_data = resp.json()

        # Step 3: Extract relevant sections from the label
        # DailyMed returns sections with title and text fields
        result = {}
        target_sections = [
            'drug interactions',
            'clinical pharmacology',
            'mechanism of action',
            'pharmacodynamics',
            'pharmacokinetics',
            'warnings and precautions',
            'contraindications'
        ]

        # Walk through all sections recursively
        def extract_sections(sections, result):
            if not sections:
                return
            for section in sections:
                title = section.get('title', '').lower().strip()
                text = section.get('text', '')

                for target in target_sections:
                    if target in title and text:
                        # Clean HTML tags from text
                        clean_text = re.sub(r'<[^>]+>', ' ', text)
                        clean_text = re.sub(r'\s+', ' ', clean_text).strip()

                        if len(clean_text) > 30:
                            # Cap at 1500 chars to keep chunks manageable
                            result[title] = clean_text[:1500]

                # Recurse into subsections
                if 'sections' in section:
                    extract_sections(section['sections'], result)

        top_sections = label_data.get('data', {}).get('sections', [])
        extract_sections(top_sections, result)

        return result if result else None

    except Exception as e:
        return None


# Test with one validation drug
print("Testing DailyMed API with Alprazolam...")
test = fetch_dailymed_data("Alprazolam")
if test:
    print(f"Sections found: {list(test.keys())}")
    for key, val in test.items():
        print(f"\n--- {key} ---")
        print(val[:300] + "..." if len(val) > 300 else val)
else:
    print("No DailyMed data returned")

In [ ]:
# ============================================
# STEP 8b: Debug DailyMed API response structure
# ============================================
#
# WHY:  The test returned no data. We need to see what the API
#       actually returns to fix our parsing logic.
# ============================================

import requests
import json

# Step 1: Check if search finds Alprazolam
search_url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
params = {'drug_name': 'Alprazolam', 'page': 1, 'pagesize': 1}

resp = requests.get(search_url, params=params, timeout=15)
print(f"Search status: {resp.status_code}")
search_data = resp.json()
print(f"Search results: {len(search_data.get('data', []))}")

if search_data.get('data'):
    set_id = search_data['data'][0].get('setid')
    print(f"Set ID: {set_id}")
    print(f"Title: {search_data['data'][0].get('title', 'N/A')}")

    # Step 2: Fetch the full label
    label_url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.json"
    resp2 = requests.get(label_url, timeout=15)
    print(f"\nLabel status: {resp2.status_code}")
    label_data = resp2.json()

    # Step 3: See what structure we get
    data = label_data.get('data', {})
    print(f"Top-level keys: {list(data.keys())}")

    sections = data.get('sections', [])
    if not sections:
        # Maybe sections are nested differently
        print(f"\nNo 'sections' key. Exploring data structure:")
        for key in data.keys():
            val = data[key]
            if isinstance(val, list):
                print(f"  {key}: list[{len(val)}]")
                if val:
                    print(f"    first item type: {type(val[0])}")
                    if isinstance(val[0], dict):
                        print(f"    first item keys: {list(val[0].keys())}")
            elif isinstance(val, dict):
                print(f"  {key}: dict with keys {list(val.keys())[:5]}")
            else:
                print(f"  {key}: {type(val).__name__} = {str(val)[:100]}")
    else:
        print(f"\nFound {len(sections)} sections:")
        for s in sections[:10]:
            title = s.get('title', s.get('name', 'NO TITLE'))
            has_text = 'text' in s
            has_subsections = 'sections' in s
            print(f"  - '{title}' (text: {has_text}, subsections: {has_subsections})")

In [ ]:
# ============================================
# STEP 8c: Fix DailyMed API - use XML endpoint
# ============================================
#
# WHY:  DailyMed's /spls/{id}.json returns 415 error.
#       The label content is only available as XML (SPL format).
#       We use the XML endpoint and parse the relevant sections.
# ============================================

import requests
import xml.etree.ElementTree as ET
import re

def fetch_dailymed_data(drug_name):
    """
    Fetch drug interaction and clinical pharmacology sections
    from DailyMed FDA labels using XML endpoint.

    Steps:
      1. Search DailyMed by drug name to get the set_id
      2. Fetch the SPL XML using that set_id
      3. Parse and extract relevant text sections
    """

    # Step 1: Search for the drug
    search_url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    params = {'drug_name': drug_name, 'page': 1, 'pagesize': 1}

    try:
        resp = requests.get(search_url, params=params, timeout=15)
        if resp.status_code != 200:
            return None

        search_data = resp.json()
        results = search_data.get('data', [])
        if not results:
            return None

        set_id = results[0].get('setid')
        if not set_id:
            return None

        # Step 2: Fetch the SPL XML
        xml_url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.xml"
        resp = requests.get(xml_url, timeout=20)
        if resp.status_code != 200:
            return None

        # Step 3: Parse XML and extract sections
        # SPL uses HL7 namespace
        root = ET.fromstring(resp.content)
        ns = {'spl': 'urn:hl7-org:v3'}

        result = {}
        target_sections = [
            'drug interactions',
            'clinical pharmacology',
            'mechanism of action',
            'pharmacodynamics',
            'pharmacokinetics',
            'warnings and precautions',
            'contraindications'
        ]

        # Find all sections with titles
        for component in root.iter('{urn:hl7-org:v3}section'):
            # Get section title
            title_elem = component.find('{urn:hl7-org:v3}title')
            if title_elem is None or title_elem.text is None:
                continue

            title = title_elem.text.strip().lower()

            # Check if this is a section we want
            for target in target_sections:
                if target in title:
                    # Extract all text content from this section
                    texts = []
                    for text_elem in component.iter('{urn:hl7-org:v3}paragraph'):
                        if text_elem.text:
                            texts.append(text_elem.text.strip())
                        # Also get tail text and nested text
                        for child in text_elem:
                            if child.text:
                                texts.append(child.text.strip())
                            if child.tail:
                                texts.append(child.tail.strip())

                    full_text = ' '.join(t for t in texts if t)
                    full_text = re.sub(r'\s+', ' ', full_text).strip()

                    if len(full_text) > 30:
                        result[title] = full_text[:1500]
                    break

        return result if result else None

    except Exception as e:
        return None


# Test with Alprazolam
print("Testing DailyMed XML endpoint with Alprazolam...")
test = fetch_dailymed_data("Alprazolam")
if test:
    print(f"Sections found: {list(test.keys())}")
    for key, val in test.items():
        print(f"\n--- {key} ---")
        print(val[:300] + "..." if len(val) > 300 else val)
else:
    print("No DailyMed data returned")

In [ ]:
# ============================================
# STEP 8d: Batch fetch DailyMed data for all drugs
# ============================================
#
# WHAT: Query DailyMed for all 4,266 drugs in our dataset.
#       Uses the XML endpoint that we validated in Step 8c.
#
# WHY:  FDA label drug interaction sections are the gold standard
#       for documented DDIs. This gives Claude the ability to compare
#       model predictions against official regulatory evidence.
#
# NOTE: DailyMed is slower than PubChem (~1-2s per request).
#       Not all drugs will have FDA labels (non-US, experimental,
#       biologics). We expect ~40-50% coverage.
#       Uses concurrent requests like PubChem to speed things up.
#       Saves checkpoint every 200 drugs.
#
# INPUT:  drug_info DataFrame with drug names
# OUTPUT: dailymed_data.json saved to Drive
# ============================================

from concurrent.futures import ThreadPoolExecutor, as_completed

DAILYMED_SAVE = f'{SAVE_PATH}/dailymed_data.json'

# Load existing progress
if os.path.exists(DAILYMED_SAVE):
    with open(DAILYMED_SAVE, 'r') as f:
        dailymed_data = json.load(f)
    print(f"Resuming: {len(dailymed_data)} drugs already fetched")
else:
    dailymed_data = {}
    print("Starting fresh DailyMed fetch")

# Build list of drugs to fetch (skip already fetched)
all_drug_names = drug_info[['drugbank_id', 'name']].to_dict('records')
remaining = [d for d in all_drug_names if d['drugbank_id'] not in dailymed_data]
print(f"Drugs to fetch: {len(remaining)} (of {len(all_drug_names)} total)")

def fetch_one_dailymed(drug_record):
    """Fetch DailyMed data for one drug. Returns (drugbank_id, result_dict or None)."""
    result = fetch_dailymed_data(drug_record['name'])
    if result:
        return (drug_record['drugbank_id'], {
            'name': drug_record['name'],
            'sections': result
        })
    return (drug_record['drugbank_id'], None)

# Run with 3 concurrent threads (DailyMed is slower, be conservative)
fetched = 0
errors = 0
start_time = time.time()
BATCH_SAVE = 200

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(fetch_one_dailymed, d): i for i, d in enumerate(remaining)}

    for future in as_completed(futures):
        dbid, result = future.result()

        if result:
            dailymed_data[dbid] = result
            fetched += 1
        else:
            errors += 1

        total_done = fetched + errors

        # Progress update every 100 drugs
        if total_done % 100 == 0:
            elapsed = time.time() - start_time
            rate = total_done / elapsed if elapsed > 0 else 1
            remaining_count = len(remaining) - total_done
            remaining_time = remaining_count / rate / 60 if rate > 0 else 0
            print(f"  Progress: {total_done}/{len(remaining)} | "
                  f"Fetched: {fetched} | Errors: {errors} | "
                  f"~{remaining_time:.1f} min remaining")

        # Save checkpoint
        if total_done % BATCH_SAVE == 0:
            with open(DAILYMED_SAVE, 'w') as f:
                json.dump(dailymed_data, f)

# Final save
with open(DAILYMED_SAVE, 'w') as f:
    json.dump(dailymed_data, f, indent=2)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} minutes")
print(f"Total drugs with DailyMed data: {len(dailymed_data)}")
print(f"Fetched this run: {fetched}")
print(f"No FDA label found: {errors}")

# Validation drug check
print("\nValidation drug coverage:")
for drug in validation_drugs:
    row = drug_info[drug_info['name'] == drug]
    if not row.empty:
        dbid = row['drugbank_id'].values[0]
        has_data = dbid in dailymed_data
        sections = list(dailymed_data[dbid]['sections'].keys()) if has_data else []
        print(f"  {drug}: {'YES' if has_data else 'NO'} - {sections}")

In [ ]:
# ============================================
# STEP 9: Build enriched chunks combining all 3 sources
# ============================================
#
# WHAT: Merge DrugBank (structured), PubChem (pharmacology text),
#       and DailyMed (FDA label) data into a single enriched
#       chunk per drug.
#
# WHY:  Our original chunks from Step 2 only had DrugBank data.
#       Now we add PubChem mechanism of action, pharmacodynamics,
#       metabolism text, plus FDA drug interactions and clinical
#       pharmacology. This gives Claude much richer context.
#
# HOW:  For each drug, we look up its PubChem CID and DrugBank ID
#       to find matching entries in pubchem_data and dailymed_data,
#       then append that text to the existing chunk.
#
# INPUT:  chunks from Step 2, pubchem_data from Step 7, dailymed_data from Step 8
# OUTPUT: enriched_chunks saved to Drive, ready for ChromaDB re-indexing
# ============================================

# Build CID lookup: drugbank_id → pubchem_cid (as string)
dbid_to_cid = {}
for _, row in mapping_with_names.iterrows():
    if pd.notna(row['pubchem_cid']):
        dbid = row.get('drug id', '')
        dbid_to_cid[dbid] = str(int(row['pubchem_cid']))

print(f"DrugBank→CID mapping: {len(dbid_to_cid)} drugs")
print(f"PubChem data available: {len(pubchem_data)} drugs")
print(f"DailyMed data available: {len(dailymed_data)} drugs")

enriched_chunks = []
stats = {'pubchem_added': 0, 'dailymed_added': 0, 'both_added': 0}

for chunk in chunks:
    dbid = chunk['drug_id']

    # Start with existing DrugBank text
    text_parts = [chunk['text']]
    has_pubchem = False
    has_dailymed = False

    # --- Add PubChem pharmacology ---
    cid = dbid_to_cid.get(dbid)
    pubchem_entry = pubchem_data.get(cid) if cid else None

    if pubchem_entry and pubchem_entry.get('data'):
        pc_sections = []
        for section_name, section_text in pubchem_entry['data'].items():
            pc_sections.append(f"{section_name}: {section_text[:500]}")

        if pc_sections:
            text_parts.append("PubChem Pharmacology:\n" + "\n".join(pc_sections))
            has_pubchem = True

    # --- Add DailyMed FDA label ---
    dm_entry = dailymed_data.get(dbid)

    if dm_entry and dm_entry.get('sections'):
        dm_sections = []
        for section_name, section_text in dm_entry['sections'].items():
            # Clean up section names (some have tabs and spaces)
            clean_name = ' '.join(section_name.split()).strip()
            dm_sections.append(f"FDA Label - {clean_name}: {section_text[:500]}")

        if dm_sections:
            text_parts.append("DailyMed FDA Label:\n" + "\n".join(dm_sections))
            has_dailymed = True

    # Track stats
    if has_pubchem:
        stats['pubchem_added'] += 1
    if has_dailymed:
        stats['dailymed_added'] += 1
    if has_pubchem and has_dailymed:
        stats['both_added'] += 1

    # Build enriched chunk
    enriched = chunk.copy()
    enriched['text'] = "\n\n".join(text_parts)
    enriched['sources'] = {
        'drugbank': True,
        'pubchem': has_pubchem,
        'dailymed': has_dailymed
    }
    enriched_chunks.append(enriched)

# Summary
print(f"\nEnriched {len(enriched_chunks)} chunks:")
print(f"  DrugBank only: {len(enriched_chunks) - stats['pubchem_added'] - stats['dailymed_added'] + stats['both_added']}")
print(f"  + PubChem: {stats['pubchem_added']}")
print(f"  + DailyMed: {stats['dailymed_added']}")
print(f"  + Both: {stats['both_added']}")

# Show example: enriched Alprazolam
for c in enriched_chunks:
    if c['name'] == 'Alprazolam':
        print(f"\n--- Alprazolam enriched chunk ---")
        print(f"Text length: {len(c['text'])} chars (was {len(chunks[0]['text'])} for Brimonidine)")
        print(f"Sources: {c['sources']}")
        print(f"\nFirst 500 chars:\n{c['text'][:500]}...")
        break

In [ ]:
import os
path = f'{SAVE_PATH}/rag_explanations_enriched.json'
print(f"Exists: {os.path.exists(path)}")
if os.path.exists(path):
    print(f"Size: {os.path.getsize(path) / 1024:.1f} KB")

Exists: True
Size: 26.0 KB


In [ ]:
# ============================================
# CHECKPOINT: Save enriched chunks to Drive
# ============================================
#
# WHY:  These enriched chunks are the final knowledge base.
#       Next step is re-indexing ChromaDB with them.
# ============================================

with open(f'{SAVE_PATH}/enriched_chunks.json', 'w') as f:
    json.dump(enriched_chunks, f)

file_size = os.path.getsize(f'{SAVE_PATH}/enriched_chunks.json') / (1024*1024)
print(f"Saved {len(enriched_chunks)} enriched chunks to {SAVE_PATH}/enriched_chunks.json")
print(f"File size: {file_size:.1f} MB")

In [ ]:
# ============================================
# Fix: Reload ChromaDB client (was overwritten by Anthropic client)
# ============================================
#
# WHY:  In Step 5 we did: client = anthropic.Anthropic(...)
#       This overwrote the ChromaDB client. We need both, so we
#       use different variable names going forward.
# ============================================

import chromadb

# ChromaDB client (rename to avoid conflict with Anthropic client)
chroma_client = chromadb.PersistentClient(path=f'{SAVE_PATH}/chromadb')

# Anthropic client (rename for clarity)
anthropic_client = client  # 'client' currently holds the Anthropic client

print(f"ChromaDB client restored")
print(f"Existing collections: {[c.name for c in chroma_client.list_collections()]}")

In [ ]:
# ============================================
# STEP 10: Re-index ChromaDB with enriched chunks
# ============================================
#
# WHAT: Delete the old collection and create a new one with the
#       enriched chunks that include PubChem and DailyMed data.
#
# WHY:  The old ChromaDB only had DrugBank text (~200-400 chars per drug).
#       The enriched chunks have up to ~5000 chars per drug with
#       pharmacology descriptions, FDA interactions, and metabolism info.
#       This means better semantic search AND richer context for Claude.
#
# NOTE: We use chroma_client (not client, which is Anthropic).
#       We also update generate_ddi_explanation to use anthropic_client.
#
# INPUT:  enriched_chunks from Step 9
# OUTPUT: Updated ChromaDB collection persisted to Drive
# ============================================

# Delete old collection and create fresh
chroma_client.delete_collection('ddi_drugs')
print("Deleted old collection")

collection = chroma_client.create_collection(
    name="ddi_drugs",
    metadata={"description": "Enriched drug profiles: DrugBank + PubChem + DailyMed"}
)
print("Created new collection")

# Index enriched chunks in batches
print(f"Indexing {len(enriched_chunks)} enriched chunks...")
start = time.time()

BATCH_SIZE = 50

for i in range(0, len(enriched_chunks), BATCH_SIZE):
    batch = enriched_chunks[i:i + BATCH_SIZE]

    # Cap text for embedding (MiniLM max ~256 tokens)
    texts = [c['text'][:2000] for c in batch]

    embeddings = embedder.encode(texts).tolist()

    metadatas = [{
        'drug_id': c['drug_id'],
        'ogb_idx': c['ogb_idx'],
        'name': c['name'],
        'type': c['type'],
        'has_cyp': len(c['cyp_enzymes']) > 0,
        'cyp_list': ', '.join(c['cyp_enzymes']) if c['cyp_enzymes'] else 'none',
        'n_targets': len(c['targets']),
        'n_enzymes': len(c['all_enzymes']),
        'has_smiles': c['smiles'] is not None,
        'has_pubchem': c['sources']['pubchem'],
        'has_dailymed': c['sources']['dailymed'],
        'full_text': c['text'][:5000]
    } for c in batch]

    ids = [c['drug_id'] for c in batch]

    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )

    if (i // BATCH_SIZE) % 20 == 0:
        print(f"  Indexed {min(i + BATCH_SIZE, len(enriched_chunks))}/{len(enriched_chunks)}...")

elapsed = time.time() - start
print(f"\nDone. Indexed {collection.count()} documents in {elapsed:.1f}s")

In [ ]:
# ============================================
# STEP 10b: Update retrieval + fix Anthropic client reference
# ============================================
#
# WHAT: Updated retrieval function that returns full enriched text,
#       plus updated generate_ddi_explanation to use anthropic_client.
#
# WHY:  We renamed clients to avoid the conflict:
#       chroma_client = ChromaDB, anthropic_client = Claude API
# ============================================

def retrieve_drug_context(drug_name_a, drug_name_b, collection, n_similar=3):
    """
    Hybrid retrieval: direct lookup for query drugs +
    semantic search for related drugs.
    Now returns enriched text with all 3 sources.
    """

    direct_a = collection.get(
        where={"name": drug_name_a},
        include=["documents", "metadatas"]
    )

    direct_b = collection.get(
        where={"name": drug_name_b},
        include=["documents", "metadatas"]
    )

    def get_full_text(result):
        if result['metadatas']:
            return result['metadatas'][0].get('full_text', result['documents'][0])
        return result['documents'][0] if result['documents'] else None

    query = f"{drug_name_a} {drug_name_b} interaction mechanism CYP"
    similar = collection.query(
        query_texts=[query],
        n_results=n_similar + 2
    )

    semantic_results = []
    for doc, meta in zip(similar['documents'][0], similar['metadatas'][0]):
        if meta['name'] not in [drug_name_a, drug_name_b]:
            semantic_results.append({'document': doc, 'metadata': meta})
        if len(semantic_results) >= n_similar:
            break

    context = {
        'drug_a': {
            'document': get_full_text(direct_a),
            'metadata': direct_a['metadatas'][0] if direct_a['metadatas'] else None
        },
        'drug_b': {
            'document': get_full_text(direct_b),
            'metadata': direct_b['metadatas'][0] if direct_b['metadatas'] else None
        },
        'similar_drugs': semantic_results
    }

    return context


def generate_ddi_explanation(drug_a, drug_b, pair_name, collection,
                              mlp_ig, graphsage_perturbation, graphsage_ig,
                              gat_exai, gat_ig, gat_perturbation, idx_to_name):
    """
    Full RAG pipeline with enriched context.
    Uses anthropic_client (not client) to avoid ChromaDB conflict.
    """

    retrieval = retrieve_drug_context(drug_a, drug_b, collection)

    exai_text = get_exai_context(pair_name, mlp_ig, graphsage_perturbation,
                                  graphsage_ig, gat_exai, gat_ig, gat_perturbation,
                                  idx_to_name)

    user_prompt = f"""Analyze the predicted drug-drug interaction between {drug_a} and {drug_b}.

## Retrieved Pharmacological Data (DrugBank + PubChem + FDA DailyMed)

### {drug_a}
{retrieval['drug_a']['document']}

### {drug_b}
{retrieval['drug_b']['document']}

## Model Explainability (EXAI) Results
The following results come from 7 GNN model variants analyzing this drug pair:

{exai_text}

## Question
Based on the pharmacological data and model explanations above, provide an evidence-based
explanation of why these models predict an interaction between {drug_a} and {drug_b}.
Highlight which models best capture the known pharmacological mechanism."""

    response = anthropic_client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}]
    )

    return {
        'explanation': response.content[0].text,
        'drug_a': drug_a,
        'drug_b': drug_b,
        'model': response.model,
        'usage': {
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens
        }
    }


# Quick test: verify enriched retrieval
context = retrieve_drug_context("Alprazolam", "Clomipramine", collection)

print(f"Drug A text length: {len(context['drug_a']['document'])} chars")
print(f"Drug A has PubChem: {context['drug_a']['metadata']['has_pubchem']}")
print(f"Drug A has DailyMed: {context['drug_a']['metadata']['has_dailymed']}")
print(f"\nDrug B text length: {len(context['drug_b']['document'])} chars")
print(f"Drug B has PubChem: {context['drug_b']['metadata']['has_pubchem']}")
print(f"Drug B has DailyMed: {context['drug_b']['metadata']['has_dailymed']}")

In [ ]:
# ============================================
# STEP 11: Re-run all 5 pairs with enriched RAG context
# ============================================
#
# WHAT: Generate new explanations using the enriched knowledge base
#       that now includes DrugBank + PubChem + DailyMed.
#
# WHY:  We want to compare these enriched explanations against
#       the DrugBank-only versions from Step 5b. The enriched
#       context should produce more detailed, clinically grounded
#       explanations with FDA-documented interactions.
#
# INPUT:  Updated collection, retrieval function, and generate function
# OUTPUT: Enriched explanations displayed and saved to Drive
# ============================================

import time
from IPython.display import display, Markdown

PAIR_INFO = {
    'Terfenadine + Alprazolam': ('Terfenadine', 'Alprazolam'),
    'Nilotinib + Dacomitinib': ('Nilotinib', 'Dacomitinib'),
    'Palonosetron + Clomipramine': ('Palonosetron', 'Clomipramine'),
    'Flunitrazepam + Alprazolam': ('Flunitrazepam', 'Alprazolam'),
    'Clomipramine + Atomoxetine': ('Clomipramine', 'Atomoxetine')
}

enriched_results = {}

for pair_name, (drug_a, drug_b) in PAIR_INFO.items():
    print(f"Generating: {pair_name}...")

    try:
        result = generate_ddi_explanation(
            drug_a=drug_a,
            drug_b=drug_b,
            pair_name=pair_name,
            collection=collection,
            mlp_ig=mlp_ig,
            graphsage_perturbation=graphsage_perturbation,
            graphsage_ig=graphsage_ig,
            gat_exai=gat_exai,
            gat_ig=gat_ig,
            gat_perturbation=gat_perturbation,
            idx_to_name=idx_to_name
        )

        enriched_results[pair_name] = result

        tokens_in = result['usage']['input_tokens']
        tokens_out = result['usage']['output_tokens']
        display(Markdown(f"""---
### {pair_name}
*Model: {result['model']} | Tokens: {tokens_in} in / {tokens_out} out*

{result['explanation']}
"""))

        time.sleep(2)

    except Exception as e:
        print(f"ERROR on {pair_name}: {e}")
        enriched_results[pair_name] = {'error': str(e)}

# Summary
total_in = sum(r['usage']['input_tokens'] for r in enriched_results.values() if 'usage' in r)
total_out = sum(r['usage']['output_tokens'] for r in enriched_results.values() if 'usage' in r)
cost = (total_in * 3 + total_out * 15) / 1_000_000

display(Markdown(f"""---
## Summary
| Metric | Value |
|--------|-------|
| Pairs processed | {sum(1 for r in enriched_results.values() if 'explanation' in r)}/5 |
| Total input tokens | {total_in:,} |
| Total output tokens | {total_out:,} |
| Estimated cost | ${cost:.4f} |
"""))

In [ ]:
# ============================================
# CHECKPOINT: Save enriched RAG explanations to Drive
# ============================================
#
# Saves both JSON (structured) and Markdown (readable).
# These are the final enriched outputs using all 3 data sources.
# ============================================

with open(f'{SAVE_PATH}/rag_explanations_enriched.json', 'w') as f:
    json.dump(enriched_results, f, indent=2)

with open(f'{SAVE_PATH}/rag_explanations_enriched.md', 'w') as f:
    for pair_name, result in enriched_results.items():
        if 'explanation' in result:
            f.write(f"# {pair_name}\n\n")
            f.write(result['explanation'])
            f.write(f"\n\n---\n\n")

print(f"Saved {len(enriched_results)} enriched explanations to:")
print(f"  - {SAVE_PATH}/rag_explanations_enriched.json")
print(f"  - {SAVE_PATH}/rag_explanations_enriched.md")

In [ ]:
import json
from IPython.display import display, Markdown

BASE = '/content/drive/MyDrive/ddi_capstone'

with open(f'{BASE}/rag_pipeline/rag_explanations_enriched.json') as f:
    results = json.load(f)

for pair_name, result in results.items():
    if 'explanation' in result:
        display(Markdown(f"### {pair_name}\n\n{result['explanation']}\n\n---"))

In [ ]:
context = retrieve_drug_context("Terfenadine", "Alprazolam", collection)

print(f"Drug A text length: {len(context['drug_a']['document'])} chars")
print(f"Has PubChem: {context['drug_a']['metadata']['has_pubchem']}")
print(f"Has DailyMed: {context['drug_a']['metadata']['has_dailymed']}")
print(f"\nFirst 300 chars:\n{context['drug_a']['document'][:300]}")

Drug A text length: 1864 chars
Has PubChem: True
Has DailyMed: False

First 300 chars:
Drug: Terfenadine (DB00342)
Type: small molecule
CYP Enzymes: Cytochrome P450 3A4, Cytochrome P450 3A7, Cytochrome P450 2D6, Cytochrome P450 3A5, Cytochrome P450 2C8, Cytochrome P450 2J2
Other Enzymes: 
Targets: Histamine H1 receptor, Muscarinic acetylcholine receptor M3, Muscarinic acetylcholine re


### Checking 3 sources

In [ ]:
pairs = [
    ("Terfenadine", "Alprazolam"),
    ("Nilotinib", "Dacomitinib"),
    ("Palonosetron", "Clomipramine"),
    ("Flunitrazepam", "Alprazolam"),
    ("Clomipramine", "Atomoxetine"),
]

for a, b in pairs:
    ctx_a = collection.get(where={"name": a}, include=["metadatas"])
    ctx_b = collection.get(where={"name": b}, include=["metadatas"])
    meta_a = ctx_a['metadatas'][0] if ctx_a['metadatas'] else {}
    meta_b = ctx_b['metadatas'][0] if ctx_b['metadatas'] else {}
    print(f"{a}: PubChem={meta_a.get('has_pubchem')}, DailyMed={meta_a.get('has_dailymed')}")
    print(f"{b}: PubChem={meta_b.get('has_pubchem')}, DailyMed={meta_b.get('has_dailymed')}")
    print()

Terfenadine: PubChem=True, DailyMed=False
Alprazolam: PubChem=True, DailyMed=True

Nilotinib: PubChem=True, DailyMed=True
Dacomitinib: PubChem=True, DailyMed=True

Palonosetron: PubChem=True, DailyMed=True
Clomipramine: PubChem=True, DailyMed=True

Flunitrazepam: PubChem=True, DailyMed=False
Alprazolam: PubChem=True, DailyMed=True

Clomipramine: PubChem=True, DailyMed=True
Atomoxetine: PubChem=True, DailyMed=True



deployment_v2.svg